# Page 8 — "Best-of-N and Verifiers": does judging beat counting, and what does it cost to find out

Page 7 kept the answer that came up most. This page keeps the answer a **verifier** likes best, and
then asks the only question that matters: does that beat counting?

It is not a rhetorical question. Best-of-N accuracy is `r x coverage(N)`, where `r` is how often the
verifier picks a correct candidate when one is in the pile. Majority voting's accuracy is a measured
number. So the reliability a verifier must reach to draw with counting is
`majority(N) / coverage(N)`, and that ratio is **the bar**: a quantity this notebook measures before
it scores anything, on the same candidates, so that every later claim is checked against a threshold
that was fixed in advance.

Three rules carry the page, and the first figure is all three on one axis:

| rule | what it knows | cost |
|---|---|---|
| one chain (plain CoT) | nothing; it is the single-sample baseline | free |
| majority vote | which answer recurs, which is page 7's rule unchanged | free |
| **best-of-N** | a verifier's judgement of each candidate | one forward pass per candidate |
| coverage (oracle) | the ground truth, so it is the ceiling all three share | free |

Two scorers from ch05, `heuristic_score` and `avg_logprob_answer`, are kept as **diagnostics**
rather than as the payoff. Neither reads whether the reasoning is right, and the table in section 11
is where that shows.

The plan, the measured bar and the go/no-go gate live in
[`docs/page_8_measured_payoff.md`](../docs/page_8_measured_payoff.md).
Writes `out/page_08/`: `traces.jsonl` (only in `GENERATE`), `scores.jsonl`, `derived.json`,
`manifest.json`.


## 0. How to run this

The modes are the phases of the experiment, not tiers of the same run.

| `RUN_MODE` | What it does | Hardware | Time |
|---|---|---|---|
| **`REPLAY`** *(default)* | every figure from the candidates and scores already on disk | **none** | ~1 min |
| `SCORE` | **phase 0.** Runs the verifier tiers over a candidate pool that already exists. No generation. This is the decision point | ~8 GB GPU | tens of minutes |
| `GENERATE` | **phase 1.** Draws page 8's own pool, then scores it | 16 GB GPU | hours |

**Run `SCORE` first and read section 12.** Phase 1 is only worth its hours if a verifier clears the
bar in phase 0, and phase 0 is free of generation because page 7 already paid for a pool.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

`REPLAY` and `SCORE` need a pool. Either run page 7 first, or point `PAGE_07_TRACES_URL` at its
published `traces.jsonl`. `GENERATE` needs neither: it makes its own.


# Part 1 — The candidates, and the bar a verifier has to clear

Loading the pool, restating page 7's vote on it, and measuring what any verifier must beat. Nothing
here loads a model and nothing here draws.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"      # "REPLAY" (no GPU) | "SCORE" (phase 0) | "GENERATE" (phase 1)
SEED = 123

# Where the candidates come from. "page_07" is the phase 0 pilot bed: a pool that has
# already been generated and paid for. "own" is page 8's own run, which GENERATE writes.
POOL_SOURCE = "own"       # "page_07" | "own"

# An item has to carry at least this many candidates at a configuration for that
# configuration to be plotted, or the N axis means a different thing on different items.
MIN_POOL = 8

# One chain length, across the whole page. Page 7's file also holds configurations at
# shorter caps, because splitting a fixed budget N ways is page 7's axis. Mixing them in
# here would confound "more candidates" and "hotter candidates" with "shorter candidates",
# which is the one thing this page must not do. None picks the deepest cap on disk.
CHAIN_CAP = None

# The N axis is built by drawing subsets from a fixed pool, so chain LENGTH is held fixed
# and the only thing changing is how many candidates the rule may look at. Anything else
# would confound "more candidates" with "shorter candidates", which is page 7's axis.
N_RESAMPLES = 64          # random subsets per (item, N); the sampling error on each point
N_BOOTSTRAP = 1000        # resamples of the ITEM set, which is what the intervals are over

# The buckets are defined by what the vote does and then measured on the same items, so
# the pile is halved: odd-indexed chains name the bucket, even-indexed ones are what the
# curves are computed from. False is the circular version, useful only to show what the
# circularity was worth.
BUCKET_HOLDOUT = True

ITEM_LIMIT = None         # e.g. 25 for a fast pass over every cell; None uses every item

# ---------------------------------------------------------------------------
# The verifiers, in the order they are tried. The first one to clear the bar is the
# page's line and the rest are diagnostics. Tiers exist because "a good enough verifier"
# is the page's premise and the cheapest sufficient one is the honest choice.
#
#   kind="yesno"  a chat model asked whether it believes the solution, read as
#                 log P(Yes) - log P(No) at one position. No generation, no verdict to
#                 parse, and a continuous score that still ranks when the model hedges.
#   kind="rm"     a trained reward model, scored by its own head. This is Cobbe et al.'s
#                 actual construction rather than a chat model imitating it.
#
# `self` is the control and is expected to fail: a model grading its own work shares
# every error mode that produced it. Its failure is a sentence on the page, not a gap.
# ---------------------------------------------------------------------------
# `solves` is that checkpoint's OWN accuracy on these same items at this same cap,
# measured by page 6. It is not decoration: a judge that solves more of the item set than
# the pile's writer does is a better GENERATOR, and beating the vote with one is a much
# weaker claim than the page's title implies. Section 4 prints the comparison.
VERIFIERS = [
    # The writer grading its own work. Same size by construction, free because the
    # checkpoint is already resident, and the control the other tiers are read against.
    dict(name="self", kind="yesno", model=None, quant=None, solves=0.43, enabled=True),
    # SMALLER than the writer, and much worse at the task: page 6 has it at 22% against
    # the 3B's 43%. This is the page's thesis in its strongest form. A model that can
    # barely solve these problems, selecting better than the crowd counts, is
    # "verification is easier than generation" demonstrated rather than quoted.
    dict(name="qwen1.5b", kind="yesno", model="Qwen/Qwen2.5-1.5B-Instruct", quant=None,
         solves=0.22, enabled=True),
    # Stronger than the writer, and the rung to read with care: it solves 60% of these
    # items alone, which is more than the 3B's vote manages at N=16. It answers "how far
    # can a better judge go" and it cannot be the page's headline without the page saying
    # that out loud.
    dict(name="qwen7b", kind="yesno", model="Qwen/Qwen2.5-7B-Instruct", quant="8bit",
         solves=0.60, enabled=True),
    # Domain specialists. Same protocol as above, so these vary the judge's TRAINING
    # DOMAIN at roughly fixed size rather than its size: math-1.5b against qwen1.5b is the
    # controlled pair, same base, same parameter count, one tuned to chat and one to
    # mathematics. `solves` is None because nobody has measured these on MATH-500 L4/5;
    # section 4b measures it rather than guessing, because figure 3's capability axis has
    # nowhere to put a judge whose own competence is unknown.
    dict(name="math1.5b", kind="yesno", model="Qwen/Qwen2.5-Math-1.5B-Instruct",
         quant=None, solves=None, enabled=True),
    # Off, deliberately. It scores 90% on MATH-500 L3 at n=20, so on this band it is
    # almost certainly a stronger generator than the 3B that wrote the pile and probably
    # than the vote. A win from it could not be read as "the specialist judged better"
    # rather than "the stronger mathematician judged better", so it costs an hour to buy
    # an ambiguous number. math1.5b is the controlled test: same base and size as the chat
    # judge already measured, differing only in what it was tuned for.
    dict(name="math7b", kind="yesno", model="Qwen/Qwen2.5-Math-7B-Instruct",
         quant="8bit", solves=None, enabled=False),
    # A trained reward model: the conceptually right answer, because it is trained to
    # verify rather than to generate, so it is not a better generator wearing a judge's
    # hat. Every tier above is a GENERATOR asked to judge, however specialised, and none of
    # them tests Cobbe et al.'s construction. An OUTCOME reward model keeps page 9's process-supervision
    # story intact; a process reward model would win more easily and spend page 9's subject
    # to do it.
    #
    # The one tier that is not a generator asked to judge. Skywork's PRM is built on
    # Qwen2.5-1.5B, the same base and size as `qwen1.5b` and `math1.5b`, so the three of
    # them differ only in what they were trained to do: chat, mathematics, and verification.
    # That is as controlled as this comparison gets.
    #
    # `generates=False` says it cannot solve anything at all, which is different from a
    # solve rate nobody measured: a model with no generative head cannot be beating the
    # vote by being a better generator, so it belongs in the blocking claim rather than
    # excluded from it.
    dict(name="skywork-prm", kind="rm", generates=False,
         model="Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B", quant=None, solves=None,
         enabled=True),
]

# Measuring a judge's own accuracy costs one greedy chain per item, which is minutes, and
# it is what turns "the specialist judged better" into a statement someone can check
# against what the specialist could do on its own.
MEASURE_JUDGE_SOLVES = True

# Not a math-tuned checkpoint, and the reason is in this repo's own ablations rather than
# in taste: experiment 2 found every Qwen2.5-Math model at 0% compliance with a plain
# instruction, and a one-word Yes/No verdict is exactly the instruction-following they are
# worst at. Their competence on these items has also never been measured here, while all
# three general checkpoints above carry a page 6 number.

# GENERATE only. Page 8's own pool: its own run_id, the deck's instrument, and a
# temperature GRID rather than a point, because the grid is the finding.
GEN = dict(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    dataset="math500_l45",
    n_items=100,
    chain_cap=2048,
    top_p=0.95,
    # (temperature, chains per item). Page 7's pilot pool is 100 items at T=0.3 and
    # 30 items at each hotter temperature, so the hot end of figure 2 is a shape rather
    # than a measurement. This grid's first job is to fix that, and its second is to give
    # best-of-N the wide pool the bar falls with. Deep where the bar is lowest, shallower
    # where it is not: the full grid at the full depth is 25,600 generations.
    # Breadth before depth, on purpose. A first pass of 16 at every temperature is what
    # fixes figure 2's weakest number -- the hot end is currently 30 items -- and it
    # completes in roughly half the run. The deepening passes then extend the pools where
    # the bar is lowest. If the run is cut short, what survives is a complete paired grid
    # rather than one very deep configuration and three empty ones.
    grid=[(0.3, 16), (0.6, 16), (0.9, 16), (1.2, 16), (0.9, 32), (1.2, 64)],
    group=8,              # chains per generate call
)

# Where REPLAY reads what it replays.
PAGE_07_TRACES_URL = None
TRACES_URL = None
SCORES_URL = None

assert RUN_MODE in ("REPLAY", "SCORE", "GENERATE"), RUN_MODE
print(f"mode={RUN_MODE}  pool={POOL_SOURCE}  "
      f"verifiers={[v['name'] for v in VERIFIERS if v['enabled']]}")


In [ ]:
import inspect
import json
import math
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    render_prompt, extract_final_candidate, grade_answer)
from reasoning_from_scratch.ch05 import heuristic_score, avg_logprob_answer

OUT_DIR = Path("out/page_08")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_07_DIR = Path("out/page_07")

P7_TRACES = PAGE_07_DIR / "traces.jsonl"
P7_DERIVED = PAGE_07_DIR / "derived.json"
OWN_TRACES = OUT_DIR / "traces.jsonl"
SCORES_PATH = OUT_DIR / "scores.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])


## 1. The pool, deduplicated on the way in

One model, one item set, one chain length, and **one configuration per temperature**. The
temperature is an axis on this page rather than a fixed choice, because it is the knob that decides
how much there is to count and how much there is to select from.

**Deduplication is not defensive tidying.** `TraceStore` has no write lock, so two processes on one
`run_id` silently produce duplicate records. Page 7's file has held 480 of them. A duplicated
candidate is counted twice by the vote and drawn twice by the subsampler, which biases every number
below, so the load asserts uniqueness rather than trusting it.


In [ ]:
def load_pool(path, url=None, label=""):
    """Every record in one trace file, keyed and deduplicated. Returns (records, run_id)."""
    if url and not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"fetching {label} from {url}")
        path.write_text(requests.get(url, timeout=600).text, encoding="utf-8")
    assert path.exists(), (
        f"no candidates at {path}. Run the notebook that makes them, or set the matching "
        f"URL in the knobs cell.")

    rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
    assert rows, f"{path} is empty"
    run_id = Counter(r["run_id"] for r in rows).most_common(1)[0][0]
    rows = [r for r in rows if r["run_id"] == run_id]

    seen, dropped = {}, 0
    for r in rows:
        k = (r["model_id"], r["config_id"], r["problem_id"], r["sample_index"])
        if k in seen:
            dropped += 1
            assert seen[k]["trace"] == r["trace"], (
                f"two DIFFERENT generations share the key {k}. That is not a duplicated "
                f"write, it is two runs sharing a run_id, and neither can be trusted.")
            continue
        seen[k] = r
    if dropped:
        print(f"  {dropped} duplicate records dropped (identical traces under identical "
              f"keys: the unlocked-TraceStore trap)")
    return list(seen.values()), run_id


if POOL_SOURCE == "page_07":
    raw, SOURCE_RUN_ID = load_pool(P7_TRACES, PAGE_07_TRACES_URL, "page 7's candidates")
    POOL_ORIGIN = "page 7's generations, re-used as the phase 0 pilot bed"
else:
    raw, SOURCE_RUN_ID = load_pool(OWN_TRACES, TRACES_URL, "page 8's own candidates")
    POOL_ORIGIN = "page 8's own generations"

MODEL_ID = Counter(r["model_id"] for r in raw).most_common(1)[0][0]
raw = [r for r in raw if r["model_id"] == MODEL_ID]
# Names everything this page writes: scores, judge competences, artifacts. Defined here
# because it is a property of the POOL, and section 4b needs it before section 5 runs.
SCORE_ID = f"p08-{SOURCE_RUN_ID}"

print(f"source run  {SOURCE_RUN_ID}")
print(f"model       {MODEL_ID}")
print(f"records     {len(raw):,}  ({POOL_ORIGIN})")


In [ ]:
# POOLS[config_id][problem_id] -> candidates, sorted by sample_index. One configuration is
# one temperature at one chain cap, which is what page 7's config_id already names.
POOLS, CONFIG_META = {}, {}
for r in raw:
    POOLS.setdefault(r["config_id"], {}).setdefault(r["problem_id"], []).append(r)
for cid in POOLS:
    for pid in POOLS[cid]:
        POOLS[cid][pid].sort(key=lambda r: r["sample_index"])

# A configuration is usable when enough items carry enough candidates. Trimming every item
# to the same depth is what makes an average across items one measurement.
usable = {}
for cid, items_at in POOLS.items():
    deep = {p: v for p, v in items_at.items() if len(v) >= MIN_POOL}
    if not deep:
        continue
    depth = min(len(v) for v in deep.values())
    usable[cid] = {p: v[:depth] for p, v in deep.items()}
    t = next(iter(deep.values()))[0]
    CONFIG_META[cid] = dict(config_id=cid, temperature=t["temperature"],
                            top_p=t.get("top_p"), chain_budget=t["chain_budget"],
                            pool_n=depth, n_items=len(deep))
POOLS = usable
assert POOLS, (f"no configuration carries {MIN_POOL} candidates on any item. Lower MIN_POOL, "
               f"or wait for the run that is making them to finish.")

# One chain length. Ranked by how much pile each cap carries, so "the deepest cap" means
# the one the page is actually about rather than whichever sorts first.
caps = Counter()
for cid, m in CONFIG_META.items():
    caps[m["chain_budget"]] += m["n_items"] * m["pool_n"]
CAP = CHAIN_CAP if CHAIN_CAP else max(caps, key=lambda c: (caps[c], c))
dropped_caps = sorted(c for c in caps if c != CAP)
POOLS = {c: v for c, v in POOLS.items() if CONFIG_META[c]["chain_budget"] == CAP}
assert POOLS, f"no configuration at cap {CAP}"

# Greedy is not a pile: N copies of one deterministic chain are one chain, and every rule
# here would score identically on it. It is kept out of the ladder and used in section 11
# as the reference the logprob diagnostic is compared against.
GREEDY_CONFIG = next((c for c in POOLS if CONFIG_META[c]["temperature"] == 0.0), None)
LADDER_CONFIGS = sorted((c for c in POOLS if c != GREEDY_CONFIG),
                        key=lambda c: CONFIG_META[c]["temperature"])
assert LADDER_CONFIGS, "every configuration on disk is greedy; there is no pile to select from"

# Two item sets, because a pilot pool is not square. A pool generated to answer page 7's
# question covers every item at its own temperature and a subset at the others, so:
#
#   ITEMS_AT[cid]  the items that configuration actually has. Figure 1 and the buckets are
#                  drawn on these, at their full strength.
#   GRID_ITEMS     the items EVERY configuration has. The temperature grid is drawn on
#                  these and on nothing else, because comparing temperatures across
#                  different problems would compare difficulty, not temperature.
ITEMS_AT = {c: sorted(POOLS[c]) for c in LADDER_CONFIGS}
GRID_ITEMS = sorted(set.intersection(*[set(v) for v in ITEMS_AT.values()]))
if ITEM_LIMIT:
    ITEMS_AT = {c: v[:ITEM_LIMIT] for c, v in ITEMS_AT.items()}
    GRID_ITEMS = [p for p in GRID_ITEMS if any(p in v for v in ITEMS_AT.values())][:ITEM_LIMIT]
    print(f"ITEM_LIMIT: {ITEM_LIMIT} items per configuration")
assert GRID_ITEMS, "the configurations share no items, so no temperature can be compared"

# The configuration the page's own figures are drawn on: the one carrying the most pile.
MAIN_CONFIG = max(LADDER_CONFIGS, key=lambda c: len(ITEMS_AT[c]) * CONFIG_META[c]["pool_n"])
ITEMS = ITEMS_AT[MAIN_CONFIG]

print(f"chain cap {CAP} tokens"
      + (f"   ({', '.join(str(c) for c in dropped_caps)} dropped: shorter chains are "
         f"page 7's axis, not this page's)" if dropped_caps else ""))
print(f"\n{'configuration':<16}{'T':>6}{'items':>7}{'pool':>6}   role")
for cid in sorted(CONFIG_META, key=lambda c: (CONFIG_META[c]["chain_budget"],
                                              CONFIG_META[c]["temperature"])):
    m = CONFIG_META[cid]
    if m["chain_budget"] != CAP:
        role = f"not used (cap {m['chain_budget']})"
    elif cid == GREEDY_CONFIG:
        role = "greedy reference, not a pile"
    else:
        role = "ladder" + (", and the page's own figures" if cid == MAIN_CONFIG else "")
    print(f"{cid:<16}{m['temperature']:>6.2f}{m['n_items']:>7}{m['pool_n']:>6}   {role}")
print(f"\n{len(GRID_ITEMS)} items shared by every configuration: the temperature grid")
print(f"{len(ITEMS)} items at {MAIN_CONFIG}: figure 1, the buckets and the worked case")
print(f"{sum(len(POOLS[c][p]) for c in LADDER_CONFIGS for p in ITEMS_AT[c]):,} candidates "
      f"in the ladder")
if len(GRID_ITEMS) < len(ITEMS):
    print(f"\nThe grid is the narrow end: {len(GRID_ITEMS)} items resolves a shape and does")
    print(f"not resolve a few points of accuracy. GENERATE fills the hot configurations out")
    print(f"to the full item set, which is most of what phase 1 is FOR.")


## 2. Page 7's vote, carried over unchanged

The vote is one of the three lines, so it is reimplemented here rather than approximated: the same
`number_only` vote key, the same equivalence relation, the same rule that a tie is not a vote. If it
drifts from page 7 the two pages report different numbers for one procedure, so section 13 checks it
against page 7's own `derived.json`.


In [ ]:
NO_ANSWER = None


def vote_key(text):
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ:
        try:
            _EQ[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ[k] = False
    return _EQ[k]


def cluster(keys):
    """Answer strings grouped into equivalence classes, in first-seen order."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


def plurality(cands):
    """The class the most candidates agree on, or None on a tie or an empty tally."""
    labels, _ = cluster([c.get("vote_key") or vote_key(c["trace"]) for c in cands])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    mc = counts.most_common()
    if not mc:
        return None
    top = mc[0][1]
    winners = [a for a, c in mc if c == top]
    return winners[0] if len(winners) == 1 else None


def vote_correct(cands):
    win = plurality(cands)
    return bool(win is not None and grade_answer(win, cands[0]["ground_truth"]))


## 3. The bar, measured before anything is scored

`bar(N) = majority(N) / coverage(N)`, the reliability a verifier must reach to draw with counting. **It is fixed here, before any verifier is loaded, so the threshold cannot then be
chosen to suit the result.**

In [ ]:
def n_axis_for(depth):
    ns = [n for n in (1, 2, 4, 8, 16, 32, 64, 128) if n <= depth]
    if ns[-1] != depth:
        ns.append(depth)
    return ns


def coverage_at(n_correct, pool_n, n):
    """P(at least one correct in a draw of n), exactly. The unbiased pass@N estimator,
    which is cheaper and less noisy than resampling for this one quantity."""
    if n_correct == 0:
        return 0.0
    if pool_n - n_correct < n:
        return 1.0
    return 1.0 - math.comb(pool_n - n_correct, n) / math.comb(pool_n, n)


def cluster_ci(per_item, reps=N_BOOTSTRAP, tag=""):
    """A 95% interval over ITEMS, which are the independent unit here.

    The draws inside one item are subsets of one pile and are heavily correlated, so a
    Wilson interval on the draw count would treat thousands of correlated draws as
    independent trials and come out several times too narrow.
    """
    vals = list(per_item)
    if not vals:
        return (0.0, 0.0)
    rng = random.Random(harness.sha256_of([str(SEED), "ci", tag, str(len(vals))])[:15])
    n, means = len(vals), []
    for _ in range(reps):
        means.append(sum(vals[rng.randrange(n)] for _ in range(n)) / n)
    means.sort()
    return (means[int(0.025 * reps)], means[int(0.975 * reps)])


def subsets_for(pid, cid, n, repeats, size):
    """The draws every selector sees at this (item, configuration, N). Exhaustive when the
    pile is small enough that resampling would just repeat the same subsets."""
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), cid, pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def bar_rows(cid, item_ids, tag=""):
    """majority, coverage and their ratio at every N, on one item set."""
    rows = []
    p1 = mean([mean([x["correct"] for x in POOLS[cid][p]]) for p in item_ids])
    for n in n_axis_for(CONFIG_META[cid]["pool_n"]):
        cov, maj = [], []
        for pid in item_ids:
            c = POOLS[cid][pid]
            cov.append(coverage_at(sum(x["correct"] for x in c), len(c), n))
            maj.append(mean([vote_correct([c[i] for i in idx])
                             for idx in subsets_for(pid, cid, n, N_RESAMPLES, len(c))]))
        m, cv = mean(maj), mean(cov)
        lo, hi = cluster_ci(maj, tag=f"bar-{tag}{cid}-{n}")
        clo, chi = cluster_ci(cov, tag=f"cov-{tag}{cid}-{n}")
        rows.append(dict(n=n, n_items=len(item_ids), pass_at_1=p1,
                         majority=m, majority_lo=lo, majority_hi=hi,
                         coverage=cv, coverage_lo=clo, coverage_hi=chi,
                         bar=(m / cv) if cv else float("nan"),
                         bar_lo=(lo / chi) if chi else float("nan"),
                         bar_hi=(hi / clo) if clo else float("nan")))
    return rows


# On each configuration's own items, which is what figure 1 is drawn from.
BAR = {cid: bar_rows(cid, ITEMS_AT[cid]) for cid in LADDER_CONFIGS}
# On the items every configuration shares, which is the only fair way to compare them.
BAR_GRID = {cid: bar_rows(cid, GRID_ITEMS, tag="grid-") for cid in LADDER_CONFIGS}

print("the bar: reliability a verifier needs to DRAW with counting\n")
for cid in LADDER_CONFIGS:
    m = CONFIG_META[cid]
    print(f"T={m['temperature']:.2f}, n={len(ITEMS_AT[cid])} items   {'N':>4}"
          f"{'one chain':>11}{'majority':>10}{'coverage':>10}{'bar':>8}")
    for r in BAR[cid]:
        print(f"{'':<24}{r['n']:>4}{r['pass_at_1']:>11.1%}{r['majority']:>10.1%}"
              f"{r['coverage']:>10.1%}{r['bar']:>8.0%}")
    print()

# "At its best" is read off each configuration's own items at its own deepest N, because
# that is the number each rule would actually be quoted at.
BEST_VOTE_CONFIG = max(LADDER_CONFIGS, key=lambda c: BAR[c][-1]["majority"])
LOWEST_BAR_CONFIG = min(LADDER_CONFIGS, key=lambda c: BAR[c][-1]["bar"])
bv, lb = BAR[BEST_VOTE_CONFIG][-1], BAR[LOWEST_BAR_CONFIG][-1]
print(f"counting is strongest at {BEST_VOTE_CONFIG}  "
      f"{bv['majority']:.0%} at N={bv['n']}, n={bv['n_items']}")
print(f"the bar is lowest at     {LOWEST_BAR_CONFIG}  "
      f"{lb['bar']:.0%} at N={lb['n']}, n={lb['n_items']} "
      f"[{lb['bar_lo']:.0%}-{lb['bar_hi']:.0%}]")
print("\nA verifier that clears the second number wins its own configuration. A verifier")
print("that beats the FIRST number wins the page, because that is counting at its best,")
print("and section 12 is where that comparison is made rather than assumed.")


# Part 2 — The verifier

The rung the page rests on. Everything above this point is free; everything below needs weights.


## 4. What a verifier is asked, and how the answer is read

The judge is put the candidate and asked one word, and the score is
`log P(Yes) - log P(No)` at the position where that word would start. One forward pass per
candidate: no sampling, no generation, and no verdict text to parse. Reading a free-text verdict
would fail exactly on the candidates where the model hedges, which are the interesting ones, and a
continuous score is what ranking needs anyway.

The judge sees the question and the candidate. It never sees the ground truth and never sees the
other candidates, which is what keeps it strictly below the oracle rung and what makes its score a
judgement rather than a vote.

**A trained reward model is scored by its own head instead**, with no prompt of ours in the way.
That is Cobbe et al.'s construction rather than a chat model imitating it, and it is the tier to
reach for if the chat judges fall short of the bar.

### Which checkpoints, and why size matters more than subject

The tiers are a **size ladder around the model that wrote the candidates**, not a hunt for the
best available judge. A judge that is simply a stronger model at the task is a confound: if it can
tell which candidate is right it can probably solve the problem, and then "selection beat
counting" and "a bigger model beat a smaller one" are the same sentence. **So the rung that
settles the claim is the one at or below the writer's size**, and section 12 reports it separately.
Each tier's own competence, and why no math-tuned checkpoint is scored, are stated beside the
tiers in the knobs cell; the ladder is defended in
[`docs/page_8_measured_payoff.md`](../docs/page_8_measured_payoff.md) §4.

In [ ]:
JUDGE_PROMPT = (
    "You are grading a proposed solution to a mathematics problem.\n\n"
    "Problem:\n{question}\n\n"
    "Proposed solution:\n{solution}\n\n"
    "Is the final answer correct? Reply with one word: Yes or No."
)
_YES_NO = {}


def yes_no_ids(tok):
    """The single tokens a verdict would start with. Asserted single, because scoring the
    first token of a two-token word would score a prefix and not a verdict."""
    key = id(tok)
    if key not in _YES_NO:
        ids = {}
        for word in ("Yes", "No"):
            enc = tok(word, add_special_tokens=False).input_ids
            assert len(enc) == 1, (
                f"{word!r} is {len(enc)} tokens on this tokenizer, so the verdict cannot be "
                f"read off one position. Pick different verdict words and record them.")
            ids[word] = enc[0]
        _YES_NO[key] = ids
    return _YES_NO[key]


@torch.inference_mode()
def yes_no_score(model, tok, question, solution):
    """log P(Yes) - log P(No) at the verdict position. Positive means the judge believes
    the solution; the magnitude is what ranks candidates."""
    ids = yes_no_ids(tok)
    chat = tok.apply_chat_template(
        [{"role": "user", "content": JUDGE_PROMPT.format(question=question,
                                                         solution=solution)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
    logits = model(**enc).logits[0, -1].float()
    lp = torch.log_softmax(logits, dim=-1)
    return float(lp[ids["Yes"]] - lp[ids["No"]])


RM_MAX_TOKENS = 8192      # our candidates run ~1,500; this only guards a pathological one


class ValueHeadRM:
    """A reward model rebuilt on the transformers implementation of its own base.

    Skywork's checkpoint ships a copy of the Qwen2 modelling code written against
    transformers 4.x. Under 5.x that copy returns **NaN hidden states** in bf16 and fp32,
    with sdpa and with eager attention: the body is broken, not merely deprecated. Scoring
    through it would have produced an AUC of exactly 0.500 from NaNs tying with each other,
    which reads exactly like a real negative result and is not one.

    The weights are fine. The checkpoint is stock Qwen2 (338 tensors, no missing and no
    unexpected keys against `Qwen2Model`) plus one `hidden_size -> 1` linear head. So the
    body is loaded with the maintained implementation and the head applied here. Same
    arithmetic, maintained kernels, and no third-party code executed at load time.

    The reward is that head at the FINAL token, which is one number for the whole solution.
    That is the outcome reading, and it is the reading page 8 wants: a process reading would
    take a value per step, and steps are page 9's subject.
    """

    def __init__(self, model_id):
        from transformers import Qwen2Config, Qwen2Model
        from huggingface_hub import snapshot_download

        path = snapshot_download(model_id)
        sd = torch.load(Path(path) / "pytorch_model.bin", map_location="cpu",
                        weights_only=True, mmap=True)
        body = {k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}
        head = {k: v for k, v in sd.items() if k.startswith("v_head.")}
        assert head, (f"{model_id} carries no v_head.* tensors, so it is not a value-head "
                      f"reward model and this loader is the wrong one for it")

        cfg = Qwen2Config.from_pretrained(path)
        self.body = Qwen2Model(cfg)
        missing, unexpected = self.body.load_state_dict(body, strict=False)
        # Strict in effect, but reported rather than raised, because the useful message is
        # WHICH tensors diverged: any divergence means the checkpoint is not stock Qwen2
        # and the transplant is silently approximating a different architecture.
        assert not missing and not unexpected, (
            f"{model_id} does not map cleanly onto transformers' Qwen2: "
            f"missing={list(missing)[:5]} unexpected={list(unexpected)[:5]}. Rebuilding it "
            f"on this body would be scoring with a different model than the one published.")

        dev = get_device()
        self.body = self.body.to(dtype=harness.pick_dtype(dev), device=dev).eval()
        # The head runs in fp32: it is one 1536-wide dot product per candidate, the cost is
        # nothing, and the original casts to the head's dtype for the same reason.
        self.w = head["v_head.summary.weight"].to(dev, torch.float32).flatten()
        self.b = head["v_head.summary.bias"].to(dev, torch.float32)
        self.device = dev
        self.model_id = model_id
        print(f"rebuilt {model_id} on transformers' Qwen2: {len(body)} body tensors, "
              f"value head {tuple(self.w.shape)}")

    @torch.inference_mode()
    def value(self, ids):
        h = self.body(**ids, use_cache=False).last_hidden_state[0, -1].float()
        return float(h @ self.w + self.b)


@torch.inference_mode()
def reward_score(model, tok, question, solution):
    """A trained reward model's own scalar, with no prompt of ours interposed.

    **This is the OUTCOME reading of the model, and deliberately so.** The head produces a
    value at every token; the forward pass pools it at the last non-padding position, which
    is one number for the whole solution. A process reading would take the value at each
    step boundary instead, and that is page 9's subject rather than this page's. Using the
    pooled scalar is what keeps best-of-N here an outcome-supervised selector, the way
    Cobbe et al. describe it.

    The chat template is the model card's, because a reward model is calibrated to the
    format it was trained on and reformatting it silently changes what is measured.
    """
    chat = tok.apply_chat_template(
        [{"role": "user", "content": question}, {"role": "assistant", "content": solution}],
        tokenize=False)
    enc = tok(chat, return_tensors="pt", add_special_tokens=False,
              truncation=True, max_length=RM_MAX_TOKENS).to(model.device)
    v = model.value(enc)
    assert v == v, ("the reward model returned NaN. That is what a modelling-code version "
                    "mismatch looks like, and it would score every candidate identically "
                    "while an AUC of 0.500 read as a finding.")
    return v


def free_vram_gb():
    """What this process could still allocate. Reported before every load, because an
    out-of-memory error names the tier that asked rather than the tier that kept."""
    if not torch.cuda.is_available():
        return float("inf")
    torch.cuda.empty_cache()
    return torch.cuda.mem_get_info()[0] / 1e9


def vram_needed_gb(spec, headroom=1.5):
    """Rough, and deliberately so: parameter count times bytes per parameter, plus room
    for one sequence of activations. It exists to fail before a load rather than during
    one, not to be exact."""
    name = (spec.get("model") or "").lower()
    m = re.search(r"(\d+(?:\.\d+)?)b", name)
    params = float(m.group(1)) if m else 3.0
    per_param = 1.0 if spec.get("quant") == "8bit" else (0.5 if spec.get("quant") == "4bit"
                                                         else 2.0)
    return params * per_param + headroom


def load_verifier(spec):
    """The writer's own checkpoint is already resident, so `self` loads nothing."""
    if spec["kind"] == "rm":
        # A reward model is not a causal LM and harness.load_model cannot carry it: its
        # head is not a vocabulary projection, and the warm-up generate() that harness runs
        # would fail on it. Loaded here instead, at the call site, where a reader looking
        # for "what is the judge, exactly" will find it. See ValueHeadRM for why this does
        # not use the checkpoint\'s own modelling code.
        from transformers import AutoTokenizer
        return ValueHeadRM(spec["model"]), AutoTokenizer.from_pretrained(spec["model"])
    dev = get_device()
    got = harness.load_model(spec["model"], device=dev, dtype=harness.pick_dtype(dev),
                             load_in_8bit=(spec.get("quant") == "8bit"), warmup_tokens=8)
    return got["model"], got["tokenizer"]


SCORE_FN = {"yesno": yes_no_score, "rm": reward_score}
ACTIVE = [v for v in VERIFIERS if v["enabled"]]
for v in ACTIVE:
    if v["name"] != "self":
        assert v["model"], (
            f"verifier {v['name']!r} is enabled with no checkpoint named. Set `model` to one "
            f"verified against the hub, or disable the tier.")
WRITER_SOLVES = next((v["solves"] for v in VERIFIERS if v["name"] == "self"), None)
print(f"{len(ACTIVE)} verifier tier(s) enabled\n")
print("  (section 4b fills in any competence that is still unknown here)\n")
print(f"  {'tier':<11}{'kind':<8}{'solves these items alone':>26}   checkpoint")
for v in ACTIVE:
    mid = MODEL_ID if v["name"] == "self" else v["model"]
    solves = "unmeasured" if v.get("solves") is None else f"{v['solves']:.0%}"
    print(f"  {v['name']:<11}{v['kind']:<8}{solves:>26}   {mid.split('/')[-1]}")
print(f"\n  the pile's writer solves {WRITER_SOLVES:.0%} and its vote at the deepest N gets "
      f"{BAR[BEST_VOTE_CONFIG][-1]['majority']:.0%}")

OUTSOLVES = [v["name"] for v in ACTIVE
             if v.get("solves") is not None and v.get("generates", True) is not False
             and v["solves"] > BAR[BEST_VOTE_CONFIG][-1]["majority"]]
if OUTSOLVES:
    print(f"\n  Read this before reading any curve. {', '.join(OUTSOLVES)} solves more of")
    print("  this item set on its own than the whole pile's vote does. Beating the vote")
    print("  with it is therefore ambiguous between 'selection beats counting' and 'a")
    print("  bigger model beats a smaller one', and the second is not this page's claim.")
    print("  The tiers that settle it are the ones at or below the writer's own size, and")
    print("  section 12 reports them separately for exactly that reason.")


## 4b. What each judge can do on its own

Figure 3 plots what a judge *unlocks* against what it can *solve*, and a judge whose own
accuracy nobody has measured has no place on that axis. Three of the tiers take their number
from page 6, which measured them on these exact items at this exact cap. The rest are measured
here: one greedy chain per item, graded by the same ch03 grader that grades everything else.

**This is not a detour.** "The specialist judged better" and "the better model judged better"
are different claims, and the only thing that separates them is a number for what the judge
could have done by itself. Without it the ablation cannot tell you which one it found, and the
gate excludes any judge carrying an unmeasured competence from the page's blocking claim.


In [ ]:
JUDGE_SOLVES_MEASURED = {}
_need = [v for v in ACTIVE if v.get("solves") is None and v["kind"] == "yesno"]

if _need and MEASURE_JUDGE_SOLVES and RUN_MODE in ("SCORE", "GENERATE"):
    _store = harness.TraceStore(OUT_DIR / "judge_solves.jsonl", SCORE_ID,
                                ("model_id", "problem_id"))
    _items = [(p, POOLS[MAIN_CONFIG][p][0]) for p in ITEMS_AT[MAIN_CONFIG]]
    for spec in _need:
        mid = spec["model"]
        todo = [(p, r) for p, r in _items if _store.get(mid, p) is None]
        got = [_store.get(mid, p) for p, _ in _items if _store.get(mid, p) is not None]
        if todo:
            print(f"\n{spec['name']}: {len(todo)} items to solve with {mid}")
            free = free_vram_gb()
            need = vram_needed_gb(spec)
            assert free > need, f"{free:.1f} GB free, {mid} wants about {need:.1f} GB"
            dev = get_device()
            _g = harness.load_model(mid, device=dev, dtype=harness.pick_dtype(dev),
                                    load_in_8bit=(spec.get("quant") == "8bit"),
                                    warmup_tokens=8)
            _m, _t = _g["model"], _g["tokenizer"]
            bar = harness.progress(todo, desc=f"solving:{spec['name']:<8}", unit="item")
            for pid, r in bar:
                # Greedy, and the same prompt the pile was written under, so this number is
                # comparable with page 6's and with the writer's own 43%.
                out = harness.generate(render_prompt(r["question"]), _m, _t,
                                       enable_thinking=False,
                                       max_new_tokens=CONFIG_META[MAIN_CONFIG]["chain_budget"])
                ans = extract_final_candidate(out["text"])
                got.append(_store.append({
                    "model_id": mid, "problem_id": pid, "extracted_answer": ans,
                    "ground_truth": r["ground_truth"],
                    "correct": bool(grade_answer(ans, r["ground_truth"])),
                    "truncated": out["truncated"], "tokens": out["tokens_generated"]}))
            del _m, _g
            harness.free_memory()
        spec["solves"] = mean([x["correct"] for x in got])
        JUDGE_SOLVES_MEASURED[spec["name"]] = dict(
            model=mid, solves=spec["solves"], n_items=len(got),
            truncated=mean([x["truncated"] for x in got]),
            ci=harness.wilson(sum(x["correct"] for x in got), len(got)))
        lo, hi = JUDGE_SOLVES_MEASURED[spec["name"]]["ci"]
        print(f"  {spec['name']}: solves {spec['solves']:.0%} of {len(got)} items "
              f"[{lo:.0%}-{hi:.0%}]")
elif _need and RUN_MODE == "REPLAY":
    # Replaying a run that measured them reads the answers back.
    _p = OUT_DIR / "judge_solves.jsonl"
    if _p.exists():
        rows = [json.loads(l) for l in _p.open(encoding="utf-8") if l.strip()]
        for spec in _need:
            mine = [r for r in rows if r["model_id"] == spec["model"]
                    and r.get("run_id") == SCORE_ID]
            if mine:
                spec["solves"] = mean([r["correct"] for r in mine])
                JUDGE_SOLVES_MEASURED[spec["name"]] = dict(
                    model=spec["model"], solves=spec["solves"], n_items=len(mine),
                    truncated=mean([r["truncated"] for r in mine]),
                    ci=harness.wilson(sum(r["correct"] for r in mine), len(mine)))

WRITER_SOLVES = next((v["solves"] for v in VERIFIERS if v["name"] == "self"), None)
_unmeasured = [v["name"] for v in ACTIVE
               if v.get("solves") is None and v.get("generates", True) is not False]
print(f"\n  {'judge':<11}{'solves these items':>20}   source")
for v in ACTIVE:
    src = ("measured here" if v["name"] in JUDGE_SOLVES_MEASURED
           else "not a generator" if v.get("generates", True) is False
           else "unmeasured" if v.get("solves") is None else "page 6")
    val = ("n/a" if v.get("generates", True) is False
           else "-" if v.get("solves") is None else f"{v['solves']:.0%}")
    print(f"  {v['name']:<11}{val:>20}   {src}")
if _unmeasured:
    print(f"\n  {', '.join(_unmeasured)} carry no measured competence, so they are kept out")
    print("  of the blocking claim in section 12 rather than assumed harmless.")


## 5. The scoring pass

Every candidate, every enabled verifier, resumable and keyed so a re-run costs only what it adds.
`REPLAY` reads `scores.jsonl` and loads nothing.

The free scorers are recomputed in every mode: `heuristic_score` is pure text analysis, so replaying
a published number it could derive itself would be trusting where it could check.


In [ ]:
SCORES = {}          # (config_id, problem_id, sample_index) -> {field: value}


def score_slot(cid, pid, si):
    return SCORES.setdefault((cid, pid, si), {})


# --- the free scorers, always recomputed -----------------------------------------
for cid in LADDER_CONFIGS:
    for pid in ITEMS_AT[cid]:
        for r in POOLS[cid][pid]:
            s = score_slot(cid, pid, r["sample_index"])
            s["heuristic"] = heuristic_score(r["trace"])
            s["n_chars"] = len(r["trace"])

# --- whatever a previous run left on disk ----------------------------------------
if SCORES_URL and not SCORES_PATH.exists():
    print(f"fetching published scores from {SCORES_URL}")
    SCORES_PATH.write_text(requests.get(SCORES_URL, timeout=600).text, encoding="utf-8")

read_back = 0
if SCORES_PATH.exists():
    for line in SCORES_PATH.open(encoding="utf-8"):
        if not line.strip():
            continue
        s = json.loads(line)
        if s.get("score_id") != SCORE_ID:
            continue
        k = (s["config_id"], s["problem_id"], s["sample_index"])
        if k not in SCORES:
            continue
        # The file stores a scorer's NAME beside its score, one row per (candidate,
        # scorer); the analysis wants one column per scorer on each candidate. This is
        # where the two shapes meet, and getting it wrong would leave REPLAY silently
        # verifier-less rather than loudly broken.
        if "scorer" in s and s.get("score") is not None:
            SCORES[k][f"v:{s['scorer']}"] = s["score"]
            read_back += 1
        if s.get("avg_logprob") is not None:
            SCORES[k]["avg_logprob"] = s["avg_logprob"]
if read_back:
    have_fields = Counter(f for v in SCORES.values() for f in v if f.startswith("v:"))
    print(f"{read_back:,} scores read from {SCORES_PATH}: "
          + ", ".join(f"{f[2:]} {n:,}" for f, n in sorted(have_fields.items())))
else:
    print(f"no scores on disk yet for {SCORE_ID}")


In [ ]:
def answer_span(trace):
    """(everything before the answer, the answer). Falls back to the whole trace when a
    candidate never opened a box, which is the case the vote calls 'no answer'."""
    i = trace.rfind("\\boxed{")
    return (trace[:i], trace[i:]) if i != -1 else ("", trace)


class _LogitsOnly:
    """A transformers model that answers like the book's model does."""

    def __init__(self, model):
        self.m = model

    def __call__(self, ids):
        return self.m(ids).logits


class _PlainEncode:
    """A tokenizer whose `.encode` adds nothing of its own. AutoTokenizer.encode adds
    special tokens by default, which would put a marker inside the scored sequence."""

    def __init__(self, tok):
        self.t = tok

    def encode(self, s):
        return self.t(s, add_special_tokens=False).input_ids


SCORED_THIS_RUN = Counter()

if RUN_MODE in ("SCORE", "GENERATE"):
    store = harness.TraceStore(SCORES_PATH, SCORE_ID,
                               ("config_id", "problem_id", "sample_index", "scorer"))
    store.report()
    work = [(cid, pid, r) for cid in LADDER_CONFIGS for pid in ITEMS_AT[cid]
            for r in POOLS[cid][pid]]

    for spec in ACTIVE:
        field = f"v:{spec['name']}"
        # Resume: adopt anything this scorer already wrote, keyed rather than searched.
        for cid, pid, r in work:
            hit = store.get(cid, pid, r["sample_index"], spec["name"])
            if hit is not None:
                SCORES[(cid, pid, r["sample_index"])][field] = hit["score"]
                if hit.get("avg_logprob") is not None:
                    SCORES[(cid, pid, r["sample_index"])]["avg_logprob"] = hit["avg_logprob"]
        todo = [w for w in work if field not in SCORES[(w[0], w[1], w[2]["sample_index"])]]
        if not todo:
            print(f"{spec['name']}: already complete")
            continue

        model_id = MODEL_ID if spec["name"] == "self" else spec["model"]
        need = vram_needed_gb(dict(spec, model=model_id))
        free = free_vram_gb()
        print(f"\n{spec['name']}: {len(todo):,} candidates to score with {model_id}")
        print(f"  {free:.1f} GB free, this tier wants about {need:.1f} GB")
        assert free > need, (
            f"{free:.1f} GB free is not enough for {model_id} at "
            f"{spec.get('quant') or 'bf16'}, which wants about {need:.1f} GB. Something is "
            f"still resident: check that the previous tier released every reference to its "
            f"weights, and that no other process holds the card.")
        vmodel, vtok = load_verifier(dict(spec, model=model_id))
        fn = SCORE_FN[spec["kind"]]

        # The writer's checkpoint doubles as the logprob diagnostic, which needs the same
        # weights and would otherwise cost a second load of the same model.
        do_logprob = (spec["name"] == "self" and spec["kind"] == "yesno")
        if do_logprob:
            lp_model, lp_tok = _LogitsOnly(vmodel), _PlainEncode(vtok)

        bar = harness.progress(todo, desc=f"{spec['name']:<12}", unit="cand")
        for cid, pid, r in bar:
            k = (cid, pid, r["sample_index"])
            rec = {"score_id": SCORE_ID, "config_id": cid, "problem_id": pid,
                   "sample_index": r["sample_index"], "scorer": spec["name"],
                   "score": fn(vmodel, vtok, r["question"], r["trace"]),
                   "model": model_id, "quant": spec.get("quant"), "kind": spec["kind"]}
            if do_logprob:
                chat = vtok.apply_chat_template(
                    [{"role": "user", "content": render_prompt(r["question"])}],
                    tokenize=False, add_generation_prompt=True)
                head, tail = answer_span(r["trace"])
                rec["avg_logprob"] = (avg_logprob_answer(lp_model, lp_tok, chat + head, tail,
                                                         device=vmodel.device)
                                      if tail.strip() else None)
                SCORES[k]["avg_logprob"] = rec["avg_logprob"]
            store.append(rec)
            SCORES[k][field] = rec["score"]
            SCORED_THIS_RUN[spec["name"]] += 1
        # Every reference, not just the obvious one. `_LogitsOnly` holds the model in
        # `.m`, so deleting `vmodel` alone leaves the weights resident and the NEXT tier
        # loads on top of them. On a 16 GB card that is the difference between the 8-bit
        # 7B fitting and an out-of-memory error two tiers later, a long way from its cause.
        # ValueHeadRM holds the body in `.body`, so the wrapper itself must go, the same
        # trap as _LogitsOnly holding the model in `.m`.
        if hasattr(vmodel, "body"):
            del vmodel.body
        del vmodel, vtok, fn
        if do_logprob:
            del lp_model, lp_tok
        harness.free_memory()
        print(f"{spec['name']}: {SCORED_THIS_RUN[spec['name']]:,} scored, "
              f"{free_vram_gb():.1f} GB free again")

# Which verifiers actually cover every candidate. A partial verifier is not plotted: a
# curve drawn from the subset it happened to reach is a different measurement.
HAVE = []
for spec in ACTIVE:
    field = f"v:{spec['name']}"
    n = sum(1 for cid in LADDER_CONFIGS for pid in ITEMS_AT[cid] for r in POOLS[cid][pid]
            if field in SCORES[(cid, pid, r["sample_index"])])
    total = sum(len(POOLS[c][p]) for c in LADDER_CONFIGS for p in ITEMS_AT[c])
    print(f"  {spec['name']:<10}{n:>8,} of {total:,} candidates scored"
          + ("   <- complete" if n == total else "   (incomplete, not plotted)"))
    if n == total:
        HAVE.append(spec)

HAVE_LOGPROB = all("avg_logprob" in SCORES[(c, p, r["sample_index"])]
                   for c in LADDER_CONFIGS for p in ITEMS_AT[c] for r in POOLS[c][p])
if not HAVE:
    print("\nNo verifier covers the pool, so the page has no best-of-N line yet. Set")
    print("RUN_MODE = 'SCORE' and run this notebook on a GPU: that is phase 0, it generates")
    print("nothing, and section 12 is the gate it decides.")


# Part 3 — What the page shows


## 6. The rules

Each takes a list of candidates and returns the index of the one it picks. That signature is the
page's whole argument: **same candidates, same compute, different answers.**

Ties among scores break toward the lowest sample index, which is arbitrary and stated rather than
hidden. The vote's ties are not broken at all, because page 7 does not break them.


In [ ]:
def sel_one_chain(cands, rng):
    """Plain CoT. One chain, chosen without looking at any of them, which is what a reader
    who runs the model once gets. Its curve is flat by construction, and that flatness is
    the thing the other two lines are measured against."""
    return rng.randrange(len(cands))


def sel_majority(cands, rng):
    """Page 7's rule. The index of a candidate carrying the winning answer, or None when
    the vote ties or nobody answered: a non-vote is not a pick."""
    win = plurality(cands)
    if win is None:
        return None
    for i, c in enumerate(cands):
        k = c.get("vote_key") or vote_key(c["trace"])
        if k is not NO_ANSWER and equivalent(k, win):
            return i
    return None


def sel_oracle(cands, rng):
    return next((i for i, c in enumerate(cands) if c["correct"]), None)


def by_score(field, ascending=False):
    """Highest score in `field` wins, ties to the lowest index. `ascending` exists for the
    heuristic, whose brevity term means its argmax is the shortest candidate."""
    sign = -1.0 if ascending else 1.0

    def pick(cands, rng):
        def key(i):
            s = SCORES[(cands[i]["config_id"], cands[i]["problem_id"],
                        cands[i]["sample_index"])].get(field)
            return (-math.inf if s is None else sign * s, -i)
        return max(range(len(cands)), key=key)
    return pick


# The three lines the page is about, plus the ceiling they share.
HEADLINE = [("one chain", "plain CoT: one sample, unexamined", sel_one_chain),
            ("majority", "keeps the answer that recurs", sel_majority)]
for spec in HAVE:
    HEADLINE.append((f"best-of-N ({spec['name']})",
                     f"keeps the candidate {spec['name']} scores highest",
                     by_score(f"v:{spec['name']}")))
HEADLINE.append(("coverage", "a correct answer is in the pile at all", sel_oracle))

# Diagnostics. Neither reads whether the reasoning is right, and section 11 is where that
# shows. They are kept off figure 1 because they are not what the page claims.
DIAGNOSTIC = [("heuristic", "ch05 heuristic_score: boxed, and short", by_score("heuristic"))]
if HAVE_LOGPROB:
    DIAGNOSTIC.append(("logprob", "ch05 avg_logprob_answer over the answer span",
                       by_score("avg_logprob")))

SELECTORS = HEADLINE + DIAGNOSTIC
print(f"{len(SELECTORS)} rules over the same candidates:")
for name, what, _ in SELECTORS:
    print(f"  {name:<22}{what}"
          + ("" if any(name == h[0] for h in HEADLINE) else "   (diagnostic)"))


In [ ]:
def run_ladder(cid, item_ids, pool_for, n_axis, selectors, tag=""):
    """Every selector over the same draws, at every N. `pool_for` returns the candidates an
    item contributes, so the buckets below can hand it a held-out half.

    Every rule sees the SAME subsets. Comparing rules across different draws would put
    sampling noise into the spread the figure reads as a difference between rules.
    """
    out = {name: [] for name, _, _ in selectors}
    picks = []
    for n in n_axis:
        per_item = {name: [] for name, _, _ in selectors}
        for pid in item_ids:
            cands_all = pool_for(pid)
            if not cands_all:
                continue
            hits = {name: [] for name, _, _ in selectors}
            for rep, idx in enumerate(
                    subsets_for(pid, cid + tag, n, N_RESAMPLES, len(cands_all))):
                cands = [cands_all[i] for i in idx]
                rng = random.Random(
                    harness.sha256_of([str(SEED), tag, cid, pid, str(n), str(rep)])[:15])
                for name, _, fn in selectors:
                    pick = fn(cands, rng)
                    ok = bool(pick is not None and cands[pick]["correct"])
                    hits[name].append(ok)
                    if n == n_axis[-1] and not tag:
                        picks.append(dict(config_id=cid, problem_id=pid, selector=name, n=n,
                                          pick=None if pick is None
                                          else cands[pick]["sample_index"], correct=ok))
            for name, _, _ in selectors:
                per_item[name].append(mean(hits[name]))
        for name, _, _ in selectors:
            vals = per_item[name]
            lo, hi = cluster_ci(vals, tag=f"{tag}{cid}{name}{n}")
            out[name].append(dict(n=n, accuracy=mean(vals), ci_lo=lo, ci_hi=hi,
                                  n_items=len(vals)))
    return out, picks


LADDERS, PICKS = {}, []
for cid in LADDER_CONFIGS:
    axis = n_axis_for(CONFIG_META[cid]["pool_n"])
    lad, pk = run_ladder(cid, ITEMS_AT[cid], lambda pid, c=cid: POOLS[c][pid], axis,
                         SELECTORS)
    LADDERS[cid] = dict(axis=axis, ladder=lad)
    PICKS.extend(pk)

# The same rules again on the items every configuration shares, at the depth every
# configuration reaches. Anything that compares one temperature to another reads THIS,
# because otherwise a temperature difference and a difference in which problems were
# asked would be the same number.
GRID_AXIS = n_axis_for(min(CONFIG_META[c]["pool_n"] for c in LADDER_CONFIGS))
GRID_N = GRID_AXIS[-1]
LADDERS_GRID = {}
for cid in LADDER_CONFIGS:
    lad, _ = run_ladder(cid, GRID_ITEMS, lambda pid, c=cid: POOLS[c][pid], GRID_AXIS,
                        HEADLINE, tag="grid-")
    LADDERS_GRID[cid] = dict(axis=GRID_AXIS, ladder=lad)
print(f"\npaired grid: every configuration re-measured on the same {len(GRID_ITEMS)} items "
      f"at N={GRID_N}")

for cid in LADDER_CONFIGS:
    print(f"\nT={CONFIG_META[cid]['temperature']:.2f}, n={len(ITEMS_AT[cid])}   {'N':>4}"
          + "".join(f"{name:>22}" for name, _, _ in SELECTORS))
    for i, n in enumerate(LADDERS[cid]["axis"]):
        print(f"        {n:>4}"
              + "".join(f"{LADDERS[cid]['ladder'][name][i]['accuracy']:>21.1%} "
                        for name, _, _ in SELECTORS))

# A consistency check that costs nothing: one chain drawn at random must land on the same
# number as the mean candidate, because it IS one candidate. If it does not, the draws are
# wrong and every curve above them is wrong with them.
for cid in LADDER_CONFIGS:
    single = mean([mean([c["correct"] for c in POOLS[cid][p]]) for p in ITEMS_AT[cid]])
    got = LADDERS[cid]["ladder"]["one chain"][0]["accuracy"]
    tol = max(0.03, 3 * 0.5 / max(len(ITEMS_AT[cid]) * N_RESAMPLES, 1) ** 0.5)
    assert abs(got - single) < tol, (
        f"{cid}: one chain at N=1 is {got:.1%} against a mean candidate of {single:.1%}; "
        f"the subsampling is not doing what it says")
print(f"\none chain at N=1 matches the mean candidate at every configuration "
      f"(tolerance {tol:.1%})")

# Having passed that check, the sampled version is replaced by the exact one. One chain
# does not depend on N -- it is pass@1 by definition -- and at the deepest N the pile has
# exactly one subset, so the sampled version is a single draw per item and visibly wobbles.
# A baseline that wobbles invites the reader to read a trend into a line that has none.
for lads, ids_for in ((LADDERS, lambda c: ITEMS_AT[c]), (LADDERS_GRID, lambda c: GRID_ITEMS)):
    for cid in LADDER_CONFIGS:
        per_item = [mean([x["correct"] for x in POOLS[cid][p]]) for p in ids_for(cid)]
        p1 = mean(per_item)
        lo, hi = cluster_ci(per_item, tag=f"p1-{cid}-{len(per_item)}")
        for pt in lads[cid]["ladder"]["one chain"]:
            pt.update(accuracy=p1, ci_lo=lo, ci_hi=hi, exact=True)
print("the one-chain line is drawn from pass@1 exactly, and is flat by construction")


## 7. Figure 1 — judge one batch two ways, and what that is worth

Two panels: one problem, then a hundred.

**Left, a real batch.** Sixteen chains from one problem, each a dot. Its **answer class** is the
row, so the vote reads vertically: the longest row wins. Its **verifier score** is the position,
so selection reads horizontally: the rightmost dot wins. Two rules, one picture, and when the
longest row does not contain the rightmost dot they disagree.

The specimen is chosen by rule, not by eye: the problem where the vote is wrong, the verifier is
right, and the wrong answer had the most votes.

**Right, the same thing at scale.** Three lines only. One chain is flat by construction, and the
distance up to best-of-N is the page's number.


In [ ]:
import matplotlib.pyplot as plt

INK = {"one chain": "#8a8578", "majority": "#3b4a6b", "coverage": "#4a7a5c",
       "correct": "#4a7a5c", "wrong": "#b23", "verifier": "#a2704a"}
BON_NAMES = [f"best-of-N ({s['name']})" for s in HAVE]
# The line the page draws: the trained verifier if one was scored, else the best judge.
HEADLINE_JUDGE = next((s for s in HAVE if s.get("generates", True) is False), None) \
    or (HAVE[0] if HAVE else None)
HEADLINE_NAME = f"best-of-N ({HEADLINE_JUDGE['name']})" if HEADLINE_JUDGE else None


def best_of_n_at(cid, n=-1, ladders=None):
    """(name, point) for the strongest best-of-N line at a configuration."""
    if not BON_NAMES:
        return None, None
    L = (ladders or LADDERS)[cid]["ladder"]
    best = max(BON_NAMES, key=lambda nm: L[nm][n]["accuracy"])
    return best, L[best][n]


# The configuration the page's figures are drawn at, chosen on the PAIRED grid so that a
# configuration cannot win by having been asked easier problems, then drawn at full
# strength on its own items. With no verifier scored there is nothing to maximise, so it
# falls back to the configuration carrying the most items.
FIG1_CONFIG = (max(LADDER_CONFIGS,
                   key=lambda c: best_of_n_at(c, ladders=LADDERS_GRID)[1]["accuracy"])
               if BON_NAMES else MAIN_CONFIG)
FIG3_CONFIG = FIG1_CONFIG
print(f"figures drawn at {FIG1_CONFIG} (n={len(ITEMS_AT[FIG1_CONFIG])}), chosen on the "
      f"paired grid")


def pick_specimen(cid):
    """The most legible problem where counting loses and selection wins.

    Chosen by rule so the figure cannot be accused of being chosen by eye. The qualifying
    condition is the page's own claim -- the vote is wrong and the verifier is right -- and
    among the problems that qualify the rule takes the SHORTEST question, breaking ties on
    the widest vote margin.

    Shortest rather than widest on purpose. Ranking by margin optimises the specimen for
    drama and lands on a five-line minimisation problem that a reader has to parse before
    they can see anything. Ranking by length optimises it for being checkable: the pick is
    a greatest-common-divisor question a reader can verify in their head, which is worth
    more to the page than two extra votes on the wrong answer. Both are rules; this one
    serves the reader."""
    if not HEADLINE_JUDGE:
        return None
    field = f"v:{HEADLINE_JUDGE['name']}"
    best = None
    for pid in ITEMS_AT[cid]:
        c = POOLS[cid][pid]
        gt = c[0]["ground_truth"]
        win = plurality(c)
        if win is None or grade_answer(win, gt):
            continue                                    # the vote was right or tied
        pick = max(c, key=lambda r: (SCORES[(cid, pid, r["sample_index"])].get(field, -math.inf),
                                     -r["sample_index"]))
        if not pick["correct"]:
            continue                                    # the verifier was wrong too
        labels, _ = cluster([x.get("vote_key") or vote_key(x["trace"]) for x in c])
        margin = Counter([l for l in labels if l is not NO_ANSWER]).most_common(1)[0][1]
        # Shortest question first, widest margin as the tiebreak.
        key = (len(c[0]["question"]), -margin)
        if best is None or key < best[0]:
            best = (key, margin, pid, win, gt)
    return None if best is None else (best[1], best[2], best[3], best[4])


SPECIMEN = pick_specimen(FIG1_CONFIG)
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.4, 4.4),
                               gridspec_kw=dict(width_ratios=[1.05, 1]))

if SPECIMEN:
    margin, pid, win, gt = SPECIMEN
    c = POOLS[FIG1_CONFIG][pid]
    field = f"v:{HEADLINE_JUDGE['name']}"
    labels, order = cluster([x.get("vote_key") or vote_key(x["trace"]) for x in c])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    rows = [a for a, _ in counts.most_common()]         # most-voted answer at the top
    ypos = {a: len(rows) - 1 - i for i, a in enumerate(rows)}
    xs = [SCORES[(FIG1_CONFIG, pid, x["sample_index"])].get(field) for x in c]
    top = max(range(len(c)), key=lambda i: (xs[i] if xs[i] is not None else -math.inf, -i))

    for i, x in enumerate(c):
        lab = labels[i]
        if lab is NO_ANSWER or xs[i] is None:
            continue
        axA.scatter(xs[i], ypos[lab] + (i % 3 - 1) * 0.11, s=46,
                    color=INK["correct"] if x["correct"] else INK["wrong"],
                    zorder=3, linewidths=0)
    axA.scatter(xs[top], ypos[labels[top]] + (top % 3 - 1) * 0.11, s=190, facecolors="none",
                edgecolors=INK["verifier"], linewidths=2, zorder=4)
    axA.set_yticks(range(len(rows)))
    axA.set_yticklabels([f"${a}$  ({counts[a]})" if len(a) < 18 else f"{a[:15]}…  ({counts[a]})"
                         for a in reversed(rows)], fontsize=9)
    axA.axhline(ypos[rows[0]], color=INK["majority"], lw=8, alpha=0.10, zorder=0)
    axA.set_xlabel(f"what {HEADLINE_JUDGE['name']} scored it")
    axA.set_ylabel("answer, and how many chains gave it")
    axA.set_title(f"the vote takes the longest row and is wrong; the verifier takes the "
                  f"rightmost dot", fontsize=9, loc="left", color="#555")
    axA.grid(axis="x", alpha=0.18)
    axA.spines[["top", "right"]].set_visible(False)
    print(f"specimen {pid}: the vote says {win!r} on {margin} of {len(c)} chains, "
          f"truth is {gt!r}")
else:
    axA.text(0.5, 0.5, "no problem where the vote loses and the verifier wins",
             ha="center", va="center", fontsize=9, color="#888")
    axA.set_xticks([]); axA.set_yticks([])

# --- right: the same claim over every problem ---------------------------------
lad = LADDERS[FIG1_CONFIG]["ladder"]
axis = LADDERS[FIG1_CONFIG]["axis"]
lines = [("one chain", INK["one chain"]), ("majority", INK["majority"])]
if HEADLINE_NAME:
    lines.append((HEADLINE_NAME, INK["verifier"]))
for name, colour in lines:
    pts = lad[name]
    xsN = [p["n"] for p in pts]
    axB.plot(xsN, [p["accuracy"] * 100 for p in pts], marker="o", ms=4, lw=2.2,
             color=colour, label=name, zorder=3)
    axB.fill_between(xsN, [p["ci_lo"] * 100 for p in pts], [p["ci_hi"] * 100 for p in pts],
                     color=colour, alpha=0.09, lw=0)
axB.plot([p["n"] for p in lad["coverage"]], [p["accuracy"] * 100 for p in lad["coverage"]],
         ls=":", lw=1.2, color=INK["coverage"], label="a correct answer is in the pile")
axB.set_xscale("log", base=2)
axB.set_xticks(axis)
axB.set_xticklabels([str(n) for n in axis])
axB.set_xlabel("candidates the rule may look at")
axB.set_ylabel("problems solved (%)")
axB.set_ylim(0, 100)
axB.grid(axis="y", alpha=0.2)
axB.spines[["top", "right"]].set_visible(False)
axB.legend(frameon=False, fontsize=8, loc="lower right")
if HEADLINE_NAME:
    axB.set_title(f"scoring {axis[-1]} candidates and keeping the best: "
                  f"{lad['one chain'][-1]['accuracy']:.0%} to "
                  f"{lad[HEADLINE_NAME][-1]['accuracy']:.0%}", fontsize=9, loc="left",
                  color="#555")
# The specimen is bundled, not just drawn: the page redraws this panel in the browser
# and cannot draw a batch it does not have. Sixteen chains at ~2.5 KB each is 40 KB, which
# is the bundled tier's job, and it lets the page open a chain the way page 7 does.
SPECIMEN_BUNDLE = None
if SPECIMEN:
    margin, pid, win, gt = SPECIMEN
    c = POOLS[FIG1_CONFIG][pid]
    field = f"v:{HEADLINE_JUDGE['name']}"
    labels, _ = cluster([x.get("vote_key") or vote_key(x["trace"]) for x in c])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    picked = max(range(len(c)),
                 key=lambda i: (SCORES[(FIG1_CONFIG, pid, c[i]["sample_index"])].get(field,
                                                                                    -math.inf),
                                -i))
    SPECIMEN_BUNDLE = dict(
        problem_id=pid, question=c[0]["question"], ground_truth=gt,
        config_id=FIG1_CONFIG, temperature=CONFIG_META[FIG1_CONFIG]["temperature"],
        n=len(c), judge=HEADLINE_JUDGE["name"],
        vote=dict(answer=win, votes=counts[labels[[i for i, l in enumerate(labels)
                                                   if l == win][0]]] if win in counts else 0,
                  correct=False),
        verifier=dict(sample_index=c[picked]["sample_index"],
                      answer=c[picked]["extracted_answer"],
                      correct=bool(c[picked]["correct"]),
                      score=SCORES[(FIG1_CONFIG, pid, c[picked]["sample_index"])].get(field)),
        candidates=[dict(sample_index=x["sample_index"],
                         answer=x["extracted_answer"],
                         answer_class=labels[i],
                         votes=counts.get(labels[i], 0),
                         correct=bool(x["correct"]),
                         score=SCORES[(FIG1_CONFIG, pid, x["sample_index"])].get(field),
                         chars=len(x["trace"]), trace=x["trace"])
                    for i, x in enumerate(c)])
    assert SPECIMEN_BUNDLE["vote"]["votes"] == margin, "the bundled tally is not the drawn one"

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_1_payoff.png", dpi=160)
plt.show()
print(f"{CONFIG_META[FIG1_CONFIG]['temperature']} temperature, "
      f"n={len(ITEMS_AT[FIG1_CONFIG])} problems, bands are 95% bootstrap intervals over items")


## 8. The temperature grid, as numbers

This was a figure until the run falsified the claim it was drawing. The pilot suggested that
diversity feeds selection and starves counting; on 100 items **coverage does not rise with
temperature at all**, so only the second half survived. What is left is real and small: the vote
weakens as the pile diversifies, the bar falls with it, and the ceiling does not move. That is a
paragraph, not a figure, and it is printed here so the page can quote it.


In [ ]:
Ts = [CONFIG_META[c]["temperature"] for c in LADDER_CONFIGS]
GRID_N = GRID_AXIS[-1]


def at_n(cid, key, n=None, table=None):
    """One entry from the PAIRED bar table at the paired depth. Every cross-temperature
    statement on this page reads this, because the configurations do not share a depth."""
    rows = (table or BAR_GRID)[cid]
    return next(r[key] for r in rows if r["n"] == (n or GRID_N))


print(f"the same {len(GRID_ITEMS)} problems at every temperature, N={GRID_N}\n")
print(f"{'T':>6}{'one chain':>11}{'majority':>10}{'coverage':>10}{'bar':>7}")
for cid in LADDER_CONFIGS:
    print(f"{CONFIG_META[cid]['temperature']:>6.2f}{at_n(cid, 'pass_at_1'):>11.1%}"
          f"{at_n(cid, 'majority'):>10.1%}{at_n(cid, 'coverage'):>10.1%}"
          f"{at_n(cid, 'bar'):>7.0%}")

COVERAGE_RISES = at_n(LADDER_CONFIGS[-1], "coverage") > at_n(LADDER_CONFIGS[0], "coverage")
print(f"\n  coverage {'rises' if COVERAGE_RISES else 'does NOT rise'} with temperature: "
      f"{at_n(LADDER_CONFIGS[0], 'coverage'):.0%} at T="
      f"{CONFIG_META[LADDER_CONFIGS[0]]['temperature']} against "
      f"{at_n(LADDER_CONFIGS[-1], 'coverage'):.0%} at T="
      f"{CONFIG_META[LADDER_CONFIGS[-1]]['temperature']}.")
if not COVERAGE_RISES:
    print("  So the extra diversity buys new ways to be wrong rather than new right answers,")
    print("  and the bar falls only because the vote collapses. That is a smaller room to win")
    print("  in, not a bigger one, and the page must not draw it as an opportunity.")


## 9. The three buckets, and the slice where selection is the only thing that can help

Every item falls into one of three, by what the vote does with it:

- **solvable** — the modal answer is correct. Page 7's regime, and counting works.
- **trap** — the modal answer is wrong and a correct answer appears at least once. The only place
  selection can beat counting.
- **hopeless** — no candidate is correct. The coverage ceiling, where no rule helps and only more
  generation or a better model can.

**The bucket is named on one half of the pile and measured on the other.** Defining the trap bucket
on the chains you then measure would guarantee that the vote looks bad there, because that is how
the bucket was chosen.


In [ ]:
# The buckets are pinned to the configuration with the most items, not to figure 1's.
# The base rate is the page's most quotable number and page 7 closes on the same one, so
# it is measured where it is best resolved.
BUCKET_CONFIG = MAIN_CONFIG
POOL_B = POOLS[BUCKET_CONFIG]
NAMING = {p: POOL_B[p][0::2] if BUCKET_HOLDOUT else POOL_B[p] for p in ITEMS}
MEASURING = {p: POOL_B[p][1::2] if BUCKET_HOLDOUT else POOL_B[p] for p in ITEMS}
BUCKET_N = min(len(v) for v in MEASURING.values())
BUCKET_AXIS = [n for n in n_axis_for(BUCKET_N)]


def bucket_of(cands):
    if vote_correct(cands):
        return "solvable"
    return "trap" if any(c["correct"] for c in cands) else "hopeless"


BUCKET = {p: bucket_of(NAMING[p]) for p in ITEMS}
BASE_RATE = {b: mean([BUCKET[p] == b for p in ITEMS])
             for b in ("solvable", "trap", "hopeless")}
FULL_PILE_RATE = {b: mean([bucket_of(POOL_B[p]) == b for p in ITEMS])
                  for b in ("solvable", "trap", "hopeless")}

print(f"at {BUCKET_CONFIG}: named by {len(next(iter(NAMING.values())))} chains, measured on "
      f"{BUCKET_N}, n={len(ITEMS)} problems")
for b, share in BASE_RATE.items():
    lo, hi = harness.wilson(round(share * len(ITEMS)), len(ITEMS))
    print(f"  {b:<10}{share:>6.0%}  {lo:.0%}-{hi:.0%}   {'#' * round(share * 34)}")
print("\n  the same three over the whole pile, which is what defining and measuring on one")
print("  set of chains would have reported: "
      + ", ".join(f"{b} {v:.0%}" for b, v in FULL_PILE_RATE.items()))
print("\n  trap = 'the right answer was there and the vote threw it away', which is what")
print("  page 7 closes on. The two numbers above are not interchangeable: page 7 quotes")
print(f"  the whole pile ({FULL_PILE_RATE['trap']:.0%}), and the held-out estimate "
      f"({BASE_RATE['trap']:.0%}) is lower because")
print("  each half is half as deep, so a correct answer is in it less often. The page")
print("  should quote the held-out number for the bucket CURVES, because that is what")
print("  they were measured on, and page 7's for the base rate it shares with page 7.")

TRAP_OK = BASE_RATE["trap"] >= 0.08
if not TRAP_OK:
    print(f"\n!! the trap bucket is {BASE_RATE['trap']:.0%}, too thin to plot a curve on")

BUCKET_LADDER = {}
for b in ("solvable", "trap", "hopeless"):
    ids = [p for p in ITEMS if BUCKET[p] == b]
    if not ids:
        continue
    lad, _ = run_ladder(BUCKET_CONFIG, ids, lambda pid: MEASURING[pid], BUCKET_AXIS,
                        HEADLINE, tag=f"bucket-{b}-")
    per_item = [mean([x["correct"] for x in MEASURING[p]]) for p in ids]
    lo, hi = cluster_ci(per_item, tag=f"p1-bucket-{b}")
    for pt in lad["one chain"]:
        pt.update(accuracy=mean(per_item), ci_lo=lo, ci_hi=hi, exact=True)
    BUCKET_LADDER[b] = dict(n_items=len(ids), share=len(ids) / len(ITEMS), ladder=lad)

for b, d in BUCKET_LADDER.items():
    print(f"\n{b}  ({d['n_items']} items, {d['share']:.0%})")
    print(f"  {'N':>4}" + "".join(f"{name:>22}" for name, _, _ in HEADLINE))
    for i, n in enumerate(BUCKET_AXIS):
        print(f"  {n:>4}" + "".join(f"{d['ladder'][name][i]['accuracy']:>21.1%} "
                                    for name, _, _ in HEADLINE))


In [ ]:
# The trap slice was a figure and is now a number: it is a real base rate, page 7 closes on
# it, and its per-bucket curves were never resolved enough to plot.
MAJORITY_DECLINES_ON_TRAPS = None
if "trap" in BUCKET_LADDER and len(BUCKET_AXIS) > 1:
    lad_t = BUCKET_LADDER["trap"]["ladder"]["majority"]
    MAJORITY_DECLINES_ON_TRAPS = bool(lad_t[-1]["accuracy"] < lad_t[0]["accuracy"]
                                      and lad_t[-1]["ci_hi"] < lad_t[0]["ci_lo"])
    print(f"on the trap slice the vote goes {lad_t[0]['accuracy']:.0%} -> "
          f"{lad_t[-1]['accuracy']:.0%}: "
          + ("declines, intervals separated" if MAJORITY_DECLINES_ON_TRAPS
             else "not resolved, the intervals overlap, so the page states the base rate "
                  "and not a slope"))


## 10. Figure 3 — what a judge is worth

One panel. Along the bottom, what each judge can solve on its own. Up the side, how often it
picks a correct candidate when one is there. The line across is the bar: clear it and selection
beats counting, fall below and it does not.

The trained verifier sits at zero on the bottom axis, and that is literal rather than a
convention: it has no generative head and cannot produce an answer to any of these problems.


In [ ]:
# --- what every rule is worth, before anything is drawn ------------------------
# Reliability is defined only where a correct candidate exists: on hopeless items every
# rule is wrong by construction and including them would compress the differences this
# figure exists to show.
reachable = {p for p in ITEMS_AT[FIG3_CONFIG]
             if any(c["correct"] for c in POOLS[FIG3_CONFIG][p])}
RELIABILITY = []
for name, what, _ in SELECTORS:
    # Per ITEM first, then bootstrap over items. The draws inside one item are subsets of
    # one pile and are not independent, and this table gets read as a ranking of judges,
    # which is exactly the reading a too-narrow interval would licence.
    per_item = {}
    for q in PICKS:
        if (q["selector"] == name and q["config_id"] == FIG3_CONFIG
                and q["problem_id"] in reachable):
            per_item.setdefault(q["problem_id"], []).append(q["correct"])
    vals = [mean(v) for v in per_item.values()]
    lo, hi = cluster_ci(vals, tag=f"rel-{FIG3_CONFIG}-{name}") if vals else (0.0, 0.0)
    pt = LADDERS[FIG3_CONFIG]["ladder"][name][-1]
    RELIABILITY.append(dict(
        selector=name, what=what, reliability=mean(vals) if vals else 0.0,
        rel_lo=lo, rel_hi=hi, n_reachable=len(vals),
        accuracy=pt["accuracy"], acc_lo=pt["ci_lo"], acc_hi=pt["ci_hi"],
        n=LADDERS[FIG3_CONFIG]["axis"][-1],
        headline=any(name == h[0] for h in HEADLINE)))
RELIABILITY_BY = {r["selector"]: r for r in RELIABILITY}
maj = RELIABILITY_BY["majority"]
bar_here = at_n(FIG3_CONFIG, "bar", LADDERS[FIG3_CONFIG]["axis"][-1], table=BAR)

JUDGE_LADDER = [
    dict(name=v["name"], solves=v.get("solves"), kind=v["kind"],
         generates=v.get("generates", True),
         params=None if v["name"] == "self" else v["model"],
         **{k: RELIABILITY_BY[f"best-of-N ({v['name']})"][k]
            for k in ("reliability", "rel_lo", "rel_hi", "accuracy", "acc_lo", "acc_hi")})
    for v in HAVE]

# Whether the ladder's ORDER is resolved, which is the reading the figure invites and the
# one it usually cannot support at these item counts.
_j = [RELIABILITY_BY[f"best-of-N ({v['name']})"] for v in HAVE]
RANKING_RESOLVED = None
if len(_j) > 1:
    _top, _second = sorted(_j, key=lambda r: -r["reliability"])[:2]
    RANKING_RESOLVED = bool(_top["rel_lo"] > _second["rel_hi"])

# --- reliability at every temperature, at a COMMON depth ------------------------
# Drawn across the whole grid rather than at one configuration, because at one
# configuration this panel's claim depends on which column was picked: at T=0.6 the four
# generator-judges look weakly ordered by capability and at T=0.3 they look inverted.
# Depth is held at the shallowest configuration's, so temperature is the only thing
# changing: reliability decays with N, and averaging over configurations of different
# depth would mix that decay into the comparison.
REL_N = GRID_AXIS[-1]
RELIABILITY_ACROSS = {}
for name, _, fn in SELECTORS:
    per_config = {}
    for cid in LADDER_CONFIGS:
        hits = []
        for pid in ITEMS_AT[cid]:
            c = POOLS[cid][pid][:REL_N]
            if not any(x["correct"] for x in c):
                continue                       # reliability is undefined where nothing is right
            i = fn(c, random.Random(0))
            hits.append(bool(i is not None and c[i]["correct"]))
        per_config[cid] = mean(hits)
    vals = list(per_config.values())
    RELIABILITY_ACROSS[name] = dict(mean=mean(vals), lo=min(vals), hi=max(vals),
                                    per_config=per_config, n=REL_N)

fig, ax = plt.subplots(figsize=(7.6, 4.6))

_pts = []
for spec in HAVE:
    r = RELIABILITY_ACROSS[f"best-of-N ({spec['name']})"]
    x = 0.0 if spec.get("generates", True) is False else spec.get("solves")
    if x is None:
        continue
    _pts.append((x * 100, r, spec))

_bar = RELIABILITY_ACROSS["majority"]
ax.axhspan(_bar["lo"] * 100, _bar["hi"] * 100, color=INK["majority"], alpha=0.07, lw=0)
ax.axhline(_bar["mean"] * 100, ls="--", lw=1.2, color=INK["majority"])
ax.text(103, _bar["mean"] * 100 + 1.0, f"the bar: {_bar['mean']:.0%}", fontsize=8,
        color=INK["majority"], ha="right")

for x, r, spec in _pts:
    trained = spec.get("generates", True) is False
    colour = INK["verifier"] if trained else "#7a7568"
    # The bar spans this judge's best and worst temperature, so a point that sits high on
    # one column and collapses on another cannot read as a steady result.
    ax.plot([x, x], [r["lo"] * 100, r["hi"] * 100], color=colour, lw=1.4, alpha=0.45,
            zorder=2, solid_capstyle="round")
    ax.scatter(x, r["mean"] * 100, s=120 if trained else 66, color=colour,
               marker="D" if trained else "o", zorder=3)
    ax.annotate(spec["name"] + ("\n(cannot solve: no\ngenerative head)" if trained else ""),
                (x, r["mean"] * 100), textcoords="offset points",
                xytext=(11, -3 if not trained else -32), fontsize=8,
                color="#333" if not trained else INK["verifier"])

ax.set_xlabel("what the judge solves on its own (%)")
ax.set_ylabel(f"picks a correct candidate when one exists (%)\nmean over four temperatures "
              f"at N={REL_N}, bar spans them")
ax.set_xlim(-5, 105)
ax.set_ylim(40, 90)
ax.grid(alpha=0.18)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("being good at the problems is not being good at judging them", fontsize=9,
             loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_judge.png", dpi=160)
plt.show()

print(f"  {'judge':<13}{'solves alone':>14}{'mean':>8}{'range':>12}"
      + "".join(f"{'T' + format(CONFIG_META[c]['temperature'], '.1f'):>7}"
                for c in LADDER_CONFIGS))
for x, r, spec in sorted(_pts, key=lambda t: t[0]):
    solves = "n/a" if spec.get("generates", True) is False else f"{spec['solves']:.0%}"
    rng = f"{r['lo']:.0%}-{r['hi']:.0%}"
    print(f"  {spec['name']:<13}{solves:>14}{r['mean']:>8.0%}{rng:>12}"
          + "".join(f"{r['per_config'][c]:>7.0%}" for c in LADDER_CONFIGS))
_b = RELIABILITY_ACROSS["majority"]
_brng = f"{_b['lo']:.0%}-{_b['hi']:.0%}"
print(f"  {'the bar':<13}{'':>14}{_b['mean']:>8.0%}{_brng:>12}"
      + "".join(f"{_b['per_config'][c]:>7.0%}" for c in LADDER_CONFIGS))
print(f"\n  Read the spread, not the order. The best solver on this ladder is the worst")
print(f"  judge on it, and the only judge whose mean clears the bar is the one that cannot")
print(f"  solve anything, because it was trained to score rather than to answer.")

if RANKING_RESOLVED is False:
    print("\n  The top two judges' intervals overlap, so this is not a ranking. What it")
    print("  shows is the absence of a trend, which does not need the ordering to be exact.")

# A verifier that cannot generate is the interesting rung, so it is stated on its own
# terms: it has no capability axis to be confounded with.
for j in [x for x in JUDGE_LADDER if x["generates"] is False]:
    beat, resolved = j["accuracy"] > maj["accuracy"], j["acc_lo"] > maj["acc_hi"]
    print(f"\n  {j['name']} was trained to score, not to answer, and cannot solve these")
    print(f"  problems at all. It picks a correct candidate {j['reliability']:.0%} of the "
          f"time one is there,")
    print(f"  against a bar of {bar_here:.0%}, for {j['accuracy']:.0%} overall against "
          f"counting's {maj['accuracy']:.0%}.")
    print("  -> " + ("clears the bar AND beats counting with the intervals separated"
                     if beat and resolved else
                     "clears the bar; ahead of counting on the estimate only"
                     if beat else "does not beat counting"))

# The cheapest judge that beats counting, measured by what it can do on its own. None of
# the non-generators are eligible: they have no solve rate to be cheap in.
_winners = [j for j in JUDGE_LADDER
            if j["accuracy"] > maj["accuracy"] and j["solves"] is not None]
CHEAPEST_SUFFICIENT = min(_winners, key=lambda j: j["solves"]) if _winners else None
if CHEAPEST_SUFFICIENT and WRITER_SOLVES:
    j = CHEAPEST_SUFFICIENT
    print(f"\n  the cheapest GENERATOR that beats counting solves {j['solves']:.0%} itself, "
          f"against the writer's {WRITER_SOLVES:.0%}")


## 11. What the diagnostics are actually ranking by

Before either ch05 scorer is called a verifier, it is worth measuring what it does, because both
turn out to be proxies for something plainer than their names suggest. This section is the reason
they are diagnostics and not the payoff.

Two numbers per scorer. **AUC** is how well the score separates a correct candidate from a wrong one
across the whole pool. **Reliability** is how often its argmax over N candidates lands on a correct
one. They can disagree sharply, and when they do it is because an argmax samples the tail of the
score distribution, where a mediocre signal is at its worst.


In [ ]:
import bisect


def auc(pairs):
    """P(score of a correct candidate > score of a wrong one), ties counted as half."""
    pos = [s for s, y in pairs if y and s is not None]
    neg = sorted(s for s, y in pairs if not y and s is not None)
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for s in pos:
        lo = bisect.bisect_left(neg, s)
        hi = bisect.bisect_right(neg, s)
        wins += lo + 0.5 * (hi - lo)
    return wins / (len(pos) * len(neg))


cands = [(c, SCORES[(FIG3_CONFIG, p, c["sample_index"])])
         for p in ITEMS_AT[FIG3_CONFIG] for c in POOLS[FIG3_CONFIG][p]]

# Cluster size is the vote's own per-candidate signal, and it is the number every verifier
# is really competing with: it is free, and on a low-temperature pile it is strong.
csize = {}
for p in ITEMS_AT[FIG3_CONFIG]:
    c = POOLS[FIG3_CONFIG][p]
    labels, _ = cluster([x.get("vote_key") or vote_key(x["trace"]) for x in c])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    for x, l in zip(c, labels):
        csize[(p, x["sample_index"])] = counts.get(l, 0)

SIGNALS = {"cluster size (the vote's own signal)":
           lambda c, s: csize[(c["problem_id"], c["sample_index"])],
           "ch05 heuristic (boxed, and short)": lambda c, s: s["heuristic"],
           "trace length": lambda c, s: s["n_chars"]}
if HAVE_LOGPROB:
    SIGNALS["ch05 avg_logprob (answer span)"] = lambda c, s: s.get("avg_logprob")
for spec in HAVE:
    SIGNALS[f"verifier: {spec['name']}"] = (
        lambda c, s, f=f"v:{spec['name']}": s.get(f))

DISCRIMINATION = []
print(f"over {len(cands):,} candidates at {FIG3_CONFIG}   (AUC 0.5 = no signal)\n")
print(f"  {'signal':<38}{'AUC':>7}{'reliability at argmax':>24}")
for label, fn in SIGNALS.items():
    a = auc([(fn(c, s), c["correct"]) for c, s in cands])
    rel = None
    for r in RELIABILITY:
        if (label.startswith("verifier: ") and r["selector"] == f"best-of-N ({label[10:]})") \
                or (label.startswith("ch05 heuristic") and r["selector"] == "heuristic") \
                or (label.startswith("ch05 avg_logprob") and r["selector"] == "logprob") \
                or (label.startswith("cluster size") and r["selector"] == "majority"):
            rel = r["reliability"]
    DISCRIMINATION.append(dict(signal=label, auc=a, reliability=rel))
    print(f"  {label:<38}{a:>7.3f}{(f'{rel:.0%}' if rel is not None else '-'):>24}")

HEURISTIC_IS_LENGTH = mean(["\\boxed{" in c["trace"] for c, _ in cands]) > 0.95
if HEURISTIC_IS_LENGTH:
    print("\n  Every candidate carries a box, so the heuristic's boxed bonus cancels and the")
    print("  rule ranks by its brevity term alone. On this pile 'heuristic score' means")
    print("  SHORTEST WINS and nothing else, which is a sharper thing to say on the page")
    print("  than 'it grades presentation'.")

GREEDY_AGREEMENT = None
if GREEDY_CONFIG and HAVE_LOGPROB:
    shared = [p for p in ITEMS_AT[FIG3_CONFIG] if p in POOLS[GREEDY_CONFIG]]
    if shared:
        pick = by_score("avg_logprob")
        agree = []
        for p in shared:
            c = POOLS[FIG3_CONFIG][p]
            g = POOLS[GREEDY_CONFIG][p][0]
            gk = g.get("vote_key") or vote_key(g["trace"])
            pk = c[pick(c, None)]
            pk = pk.get("vote_key") or vote_key(pk["trace"])
            agree.append(bool(gk and pk and equivalent(gk, pk)))
        GREEDY_AGREEMENT = dict(
            config_id=GREEDY_CONFIG, n_items=len(shared), answer_agreement=mean(agree),
            greedy_accuracy=mean([POOLS[GREEDY_CONFIG][p][0]["correct"] for p in shared]))
        print(f"\n  the logprob pick gives the same answer as the greedy chain on "
              f"{GREEDY_AGREEMENT['answer_agreement']:.0%} of items, and greedy alone solves "
              f"{GREEDY_AGREEMENT['greedy_accuracy']:.0%}")
        if GREEDY_AGREEMENT["answer_agreement"] > 0.8:
            print("  At that rate the logprob rung is largely recovering the greedy chain, so")
            print("  any gain it shows is mostly 'greedy beats a sample' rather than 'the")
            print("  model can tell which of its answers is right'. The page may not call it")
            print("  a confidence signal without saying this.")


## 11b. The paired tests

Every comparison on this page is between two rules reading **the same candidates on the same
problems**, so the right test is a paired one. Marginal confidence intervals are not: two
overlapping intervals can still hide a large, consistent, per-item difference, and two
separated ones can be produced by an effect on a handful of items. Page 7 uses an exact
McNemar over discordant items for exactly this reason, and page 8 uses the same function.

Two comparisons, and they answer different questions.

**Against one chain.** This is the comparison ch05 makes: sample N, score them, keep the best,
against just taking one. `self_consistency_vote` lives in ch04 and is not part of it. This is
what "best-of-N works" means, and it is the page's headline.

**Against the vote.** This one is ours. The book introduces counting and selection as
successive techniques and never sets them against each other; doing it on identical candidates
is what page 7 makes possible. It is reported here whatever it says, and it does not gate the
page, because it is a harder question than the one the page claims to answer.

The baseline is candidate index 0, a single fixed chain, rather than an average over draws: a
paired test needs one outcome per item, and averaging would compare a rule against a number no
reader could have obtained by running the model once.


In [ ]:
PAIRED_CONFIG = FIG3_CONFIG
_cands = POOLS[PAIRED_CONFIG]
_ids = ITEMS_AT[PAIRED_CONFIG]

# One outcome per item, on the whole pile, no subsampling: every rule here is
# deterministic given the pile, so there is nothing to average over.
_rng = random.Random(harness.sha256_of([str(SEED), "paired"])[:15])
OUTCOME = {}
for name, _, fn in SELECTORS:
    if name == "one chain":
        continue
    OUTCOME[name] = {}
    for pid in _ids:
        c = _cands[pid]
        i = fn(c, _rng)
        OUTCOME[name][pid] = bool(i is not None and c[i]["correct"])
# The baseline a reader would actually get: run the model once.
OUTCOME["one chain"] = {pid: bool(_cands[pid][0]["correct"]) for pid in _ids}

POOL_N_HERE = CONFIG_META[PAIRED_CONFIG]["pool_n"]
PAIRED = {"config_id": PAIRED_CONFIG, "n": POOL_N_HERE, "n_items": len(_ids),
          # The baseline the paired test actually used: chain 0, one fixed chain. It is
          # NOT pass@1, which averages over all of them, and the two differ by a couple of
          # points. Anything quoting "against one chain" has to quote this number.
          "baseline": "candidate index 0, one fixed chain per problem",
          "baseline_accuracy": mean([bool(_cands[pid][0]["correct"]) for pid in _ids]),
          "vs_one_chain": [], "vs_majority": []}

base = OUTCOME["one chain"]
print(f"ch05's comparison: best-of-{POOL_N_HERE} against one chain, "
      f"{len(_ids)} problems at T={CONFIG_META[PAIRED_CONFIG]['temperature']}\n")
print(f"  {'rule':<24}{'accuracy':>10}{'delta':>9}{'fixed':>7}{'broke':>7}{'p':>9}")
print(f"  {'one chain (baseline)':<24}{mean(list(base.values())):>10.1%}")
for name, _, _ in SELECTORS:
    if name in ("one chain", "coverage"):
        continue
    broke, fixed, pv = harness.mcnemar(base, OUTCOME[name])
    acc = mean(list(OUTCOME[name].values()))
    row = dict(selector=name, accuracy=acc, delta_pts=(acc - mean(list(base.values()))) * 100,
               fixed=fixed, broke=broke, p=pv, resolved=bool(pv < 0.05 and fixed > broke))
    PAIRED["vs_one_chain"].append(row)
    print(f"  {name:<24}{acc:>10.1%}{row['delta_pts']:>+8.1f}{fixed:>7}{broke:>7}{pv:>9.4f}"
          + ("  *" if row["resolved"] else ""))
print("\n  * = the rule beats one chain and the paired test resolves it")

print(f"\nand the comparison the book does not make: against the vote on the same candidates\n")
print(f"  {'rule':<24}{'accuracy':>10}{'delta':>9}{'fixed':>7}{'broke':>7}{'p':>9}")
print(f"  {'majority vote':<24}{mean(list(OUTCOME['majority'].values())):>10.1%}")
for name, _, _ in SELECTORS:
    if not name.startswith("best-of-N"):
        continue
    broke, fixed, pv = harness.mcnemar(OUTCOME["majority"], OUTCOME[name])
    acc = mean(list(OUTCOME[name].values()))
    row = dict(selector=name, accuracy=acc,
               delta_pts=(acc - mean(list(OUTCOME["majority"].values()))) * 100,
               fixed=fixed, broke=broke, p=pv, resolved=bool(pv < 0.05 and fixed > broke))
    PAIRED["vs_majority"].append(row)
    print(f"  {name:<24}{acc:>10.1%}{row['delta_pts']:>+8.1f}{fixed:>7}{broke:>7}{pv:>9.4f}"
          + ("  *" if row["resolved"] else ""))

BEST_VS_ONE = max(PAIRED["vs_one_chain"], key=lambda r: r["accuracy"]) \
    if PAIRED["vs_one_chain"] else None
BEST_VS_MAJ = max(PAIRED["vs_majority"], key=lambda r: r["accuracy"]) \
    if PAIRED["vs_majority"] else None


## 11c. Figure 2 — what each rule fixed, and what it broke

The page's evidence, on one axis. Every rule is measured against the same baseline of one chain
on the same hundred problems, so each row is a paired count: **problems it fixed to the right of
zero, problems it broke to the left.** Rows are ordered by net gain, and the exact p-value sits
at the end of each.

In [ ]:
_rows = sorted(PAIRED["vs_one_chain"], key=lambda r: r["fixed"] - r["broke"])
fig, ax = plt.subplots(figsize=(9.0, 0.52 * len(_rows) + 1.9))

for i, r in enumerate(_rows):
    ax.barh(i, r["fixed"], color=INK["correct"], height=0.6, zorder=3)
    ax.barh(i, -r["broke"], color=INK["wrong"], height=0.6, zorder=3)
    ax.text(r["fixed"] + 0.7, i, f"{r['fixed'] - r['broke']:+d} net", va="center",
            fontsize=8, color="#333")
    ax.text(-r["broke"] - 0.7, i, f"p={r['p']:.3f}" + ("  *" if r["resolved"] else ""),
            va="center", ha="right", fontsize=8,
            color="#333" if r["resolved"] else "#999")

ax.axvline(0, color="#333", lw=1)
ax.set_yticks(range(len(_rows)))
ax.set_yticklabels([r["selector"] for r in _rows], fontsize=9)
_lim = max(max(r["fixed"] for r in _rows), max(r["broke"] for r in _rows)) + 10
ax.set_xlim(-_lim, _lim)
_ticks = [t for t in range(-30, 31, 5) if abs(t) <= _lim]
ax.set_xticks(_ticks)
ax.set_xticklabels([str(abs(t)) for t in _ticks], fontsize=8)
ax.set_xlabel("problems, against taking one chain")
ax.text(_lim * 0.5, len(_rows) - 0.3, "fixed", ha="center", fontsize=9,
        color=INK["correct"], fontweight="bold")
ax.text(-_lim * 0.5, len(_rows) - 0.3, "broke", ha="center", fontsize=9,
        color=INK["wrong"], fontweight="bold")
ax.set_ylim(-0.7, len(_rows) - 0.05)
ax.grid(axis="x", alpha=0.15)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"* = the paired test resolves it.  n={PAIRED['n_items']} problems, "
             f"N={PAIRED['n']}, exact McNemar over discordant items",
             fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_paired.png", dpi=160)
plt.show()


## 12. The gate

**Does best-of-N beat counting, and beat one chain?**

The comparison is deliberately unkind to this page: best-of-N at *its* best configuration against
majority voting at *counting's* best configuration. Beating the vote only where the vote has been
starved of a crowd would be a rigged win, and this cell is what stops the page claiming one. The
whole temperature grid is printed above it, so a reader can see what was chosen from.


In [ ]:
def interval(cid, name, ladders=None, n=-1):
    p = (ladders or LADDERS)[cid]["ladder"][name][n]
    axis = (ladders or LADDERS)[cid]["axis"]
    return dict(config_id=cid, selector=name, n=axis[n], accuracy=p["accuracy"],
                ci=[p["ci_lo"], p["ci_hi"]])


# Counting at its literal best: its strongest configuration, at that configuration's
# deepest N, on that configuration's own items. This is the hard opponent, and picking a
# softer one is exactly the way this page could fool itself.
maj = interval(BEST_VOTE_CONFIG, "majority")
cot = max((dict(config_id=c, accuracy=BAR[c][0]["pass_at_1"]) for c in LADDER_CONFIGS),
          key=lambda d: d["accuracy"])
GATE = dict(majority_at_its_best=maj, one_chain_at_its_best=cot, best_of_n=None,
            paired=None)

print(f"counting at its best    {maj['config_id']}  N={maj['n']}  "
      f"n={len(ITEMS_AT[maj['config_id']])}   {maj['accuracy']:.1%} "
      f"[{maj['ci'][0]:.1%}-{maj['ci'][1]:.1%}]")
print(f"one chain at its best   {cot['config_id']}"
      f"{'':>{max(0, len(maj['config_id']) - len(cot['config_id']))}}"
      f"                    {cot['accuracy']:.1%}")

BEATS_COUNTING = BEATS_COT = False
if BON_NAMES:
    cfg, nm = max(((c, nm) for c in LADDER_CONFIGS for nm in BON_NAMES),
                  key=lambda k: LADDERS[k[0]]["ladder"][k[1]][-1]["accuracy"])
    bon = interval(cfg, nm)
    bon["verifier"] = nm[nm.find("(") + 1:-1]
    GATE["best_of_n"] = bon
    BEATS_COUNTING = bool(bon["accuracy"] > maj["accuracy"] and bon["ci"][0] > maj["ci"][1])
    BEATS_COT = bool(bon["accuracy"] > cot["accuracy"])
    print(f"best-of-N at its best   {cfg}  N={bon['n']}  n={len(ITEMS_AT[cfg])}   "
          f"{bon['accuracy']:.1%} [{bon['ci'][0]:.1%}-{bon['ci'][1]:.1%}]   ({nm})")

    # The same comparison restricted to verifiers that are NOT stronger generators than
    # the model that wrote the pile. This is the number the page's title can honestly
    # quote, and it is reported whether or not it is the best one.
    # An unmeasured judge is EXCLUDED here, not admitted. Treating "we never measured what
    # this model can do" as "it is not a stronger generator" would let the page's blocking
    # claim rest on exactly the confound the claim exists to rule out.
    clean_names = [f"best-of-N ({v['name']})" for v in HAVE
                   if v.get("generates", True) is False
                   or (v.get("solves") is not None and WRITER_SOLVES is not None
                       and v["solves"] <= WRITER_SOLVES)]
    if clean_names:
        ccfg, cnm = max(((c, n2) for c in LADDER_CONFIGS for n2 in clean_names),
                        key=lambda k: LADDERS[k[0]]["ladder"][k[1]][-1]["accuracy"])
        clean = interval(ccfg, cnm)
        GATE["best_of_n_no_stronger_generator"] = clean
        BEATS_COUNTING_CLEAN = bool(clean["accuracy"] > maj["accuracy"]
                                    and clean["ci"][0] > maj["ci"][1])
        print(f"  ... of which the best verifier that is NOT a stronger generator than the "
              f"writer:\n      {ccfg}  N={clean['n']}   {clean['accuracy']:.1%} "
              f"[{clean['ci'][0]:.1%}-{clean['ci'][1]:.1%}]   ({cnm})   "
              + ("CLEARS" if BEATS_COUNTING_CLEAN else "does not clear"))
    else:
        BEATS_COUNTING_CLEAN = False
        print("  ... and every scored verifier is a stronger generator than the writer, so")
        print("      none of them settles the page's claim on its own.")
    print(f"\n  vs counting  {(bon['accuracy'] - maj['accuracy']) * 100:+.1f} pts   "
          + ("CLEARS: the intervals are separated" if BEATS_COUNTING
             else "does NOT clear"))
    print(f"  vs one chain {(bon['accuracy'] - cot['accuracy']) * 100:+.1f} pts   "
          + ("clears" if BEATS_COT else "does NOT clear"))

    # The paired reading, on the items every configuration shares. It answers a narrower
    # question than the one above and answers it without the item sets differing.
    pm = max((interval(c, "majority", LADDERS_GRID) for c in LADDER_CONFIGS),
             key=lambda d: d["accuracy"])
    pb = max((interval(c, nm2, LADDERS_GRID) for c in LADDER_CONFIGS for nm2 in BON_NAMES),
             key=lambda d: d["accuracy"])
    GATE["paired"] = dict(n_items=len(GRID_ITEMS), n=GRID_N, majority=pm, best_of_n=pb,
                          delta_pts=(pb["accuracy"] - pm["accuracy"]) * 100,
                          separated=bool(pb["ci"][0] > pm["ci"][1]))
    print(f"\n  paired on the same {len(GRID_ITEMS)} items at N={GRID_N}: "
          f"counting {pm['accuracy']:.1%} ({pm['config_id']}) against best-of-N "
          f"{pb['accuracy']:.1%} ({pb['config_id']}), "
          f"{GATE['paired']['delta_pts']:+.1f} pts")

    if not BEATS_COUNTING:
        wide = bon["ci"][1] - bon["ci"][0] > 0.20
        print("\n  The next moves, in order, all measured rather than assumed:")
        if wide:
            print("   0. RESOLUTION FIRST. Best-of-N's own interval is wider than 20 points,")
            print("      so this is not yet a negative result. The configuration it won on")
            print(f"      carries {len(ITEMS_AT[cfg])} items; GENERATE fills it to the full")
            print("      set, and that is the cheapest thing on this list to fix.")
        print("   1. the next verifier tier in the knobs cell (a trained reward model)")
        print("   2. more N: the bar falls as coverage climbs and the vote saturates")
        print("   3. a hotter configuration, which lowers the bar (figure 2, right panel)")
        print("  If none of them clears it, the honest page says selection did not beat")
        print("  counting on this instrument, and shows the trap slice where it does.")
else:
    BEATS_COUNTING_CLEAN = False
    print("\nno verifier is scored, so the gate is not evaluated. Run RUN_MODE = 'SCORE'.")

GATE.update(beats_counting=BEATS_COUNTING, beats_one_chain=BEATS_COT)


## 13. Agreement with page 7

The vote at page 7's own configuration must reproduce page 7's number, because it is the same rule
on the same candidates. A disagreement means one of the two implementations has drifted, and it
would show up as two numbers for one procedure across a page break.

When page 8 has generated its own pool the check is weaker but still worth making: two independent
samples from one distribution should agree inside their intervals.


In [ ]:
PAGE_7_AGREEMENT = None
P7 = json.loads(P7_DERIVED.read_text()) if P7_DERIVED.exists() else None
if P7 is not None and "handoff" in P7:
    p7 = P7["handoff"]
    cid = P7.get("pool", {}).get("config_id")
    if cid in LADDERS:
        ours = LADDERS[cid]["ladder"]["majority"][-1]["accuracy"]
        cov_ours = LADDERS[cid]["ladder"]["coverage"][-1]["accuracy"]
        same_pool = POOL_SOURCE == "page_07"
        PAGE_7_AGREEMENT = dict(
            config_id=cid, n=LADDERS[cid]["axis"][-1], same_candidates=same_pool,
            page_07_vote=p7["plurality_accuracy"], page_08_vote=ours,
            page_07_coverage=p7["coverage"], page_08_coverage=cov_ours,
            vote_delta_pts=(ours - p7["plurality_accuracy"]) * 100)
        print(f"  {'':<14}{'page 7':>10}{'page 8':>10}{'delta':>9}")
        print(f"  {'the vote':<14}{p7['plurality_accuracy']:>10.1%}{ours:>10.1%}"
              f"{PAGE_7_AGREEMENT['vote_delta_pts']:>+9.1f}")
        print(f"  {'coverage':<14}{p7['coverage']:>10.1%}{cov_ours:>10.1%}"
              f"{(cov_ours - p7['coverage']) * 100:>+9.1f}")
        if same_pool and abs(PAGE_7_AGREEMENT["vote_delta_pts"]) > 0.5:
            print("\n!! Same rule, same candidates, two answers. One of the two vote")
            print("   implementations has drifted; fix it before either page ships.")
    else:
        print(f"page 7 names {cid} as its pool and this notebook has no ladder there")
else:
    print("out/page_07/derived.json not found or has no handoff; the cross-check is skipped")


## 14. The worked case

One problem where a rule picked a wrong candidate over a correct one that was sitting beside it.
Shown rather than asserted: the
reader can read both chains and see what the rule rewarded.


In [ ]:
WORKED = {}
for rule in ["heuristic"] + BON_NAMES:
    fn = dict((n, f) for n, _, f in SELECTORS)[rule]
    cases = []
    for pid in ITEMS_AT[FIG3_CONFIG]:
        c = POOLS[FIG3_CONFIG][pid]
        i = fn(c, random.Random(0))
        if i is None or c[i]["correct"]:
            continue
        right = [x for x in c if x["correct"]]
        if not right:
            continue
        # The most legible specimen: the correct candidate the rule ranked furthest down.
        best_right = max(right, key=lambda x: len(x["trace"]))
        cases.append((len(best_right["trace"]) - len(c[i]["trace"]), pid, i,
                      best_right["sample_index"]))
    if not cases:
        print(f"{rule}: no problem where it picked a wrong candidate over a correct one")
        continue
    cases.sort(reverse=True)
    _, pid, i, ri = cases[0]
    c = POOLS[FIG3_CONFIG][pid]
    picked = c[i]
    passed = next(x for x in c if x["sample_index"] == ri)
    WORKED[rule] = dict(
        problem_id=pid, question=picked["question"], ground_truth=picked["ground_truth"],
        n_cases=len(cases),
        picked=dict(sample_index=picked["sample_index"], answer=picked["extracted_answer"],
                    correct=False, chars=len(picked["trace"]), trace=picked["trace"]),
        passed_over=dict(sample_index=ri, answer=passed["extracted_answer"], correct=True,
                         chars=len(passed["trace"]), trace=passed["trace"]))
    print(f"\n{'=' * 96}\n{rule}: {len(cases)} problems show this shape; the widest is below")
    print(textwrap.fill("Q: " + picked["question"], 92))
    print("ground truth:", picked["ground_truth"])
    for label, x in (("it picked", picked), ("it passed over", passed)):
        s = SCORES[(FIG3_CONFIG, pid, x["sample_index"])]
        extra = ""
        for f in (k for k in s if k.startswith("v:")):
            extra += f"   {f[2:]} {s[f]:+.2f}"
        print(f"\n  {label}: chain {x['sample_index']}   {len(x['trace'])} chars   "
              f"heuristic {s['heuristic']:.3f}{extra}   answer {x['extracted_answer']!r}   "
              f"{'CORRECT' if x['correct'] else 'wrong'}")
        print(textwrap.fill(x["trace"].replace("\n", " "), 92, initial_indent="    ",
                            subsequent_indent="    ")[:1200])


# Part 4 — Generation, and the artifacts

`GENERATE` is phase 1 and runs only after the gate in section 12 has been cleared. It is placed
last on purpose: everything above it works on a pool that already exists, so a reader can run this
notebook end to end without a GPU and still see what the experiment is.


## 15. Phase 1 — page 8's own pool

Its own `run_id`, the deck's instrument, and a temperature grid rather than a point.

**Independent requests, not one request for N answers.** Candidates listed inside a single response
condition on each other, which is precisely the diversity collapse page 7 is about, and it would
invalidate every curve here. `num_return_sequences` from one prompt is independent sampling and is
what this uses.

The sampler is stated here rather than imported, because what a candidate *is* is the claim this
page rests on, and a reader looking for it should find it on the page that depends on it.


In [ ]:
def config_id(temperature, cap):
    return f"t{temperature:.2f}-c{cap}"


def group_seed(problem_id, cid, group_index):
    """A stable seed per (problem, configuration, group), recorded on every record in the
    group, so a group regenerates exactly."""
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


@torch.inference_mode()
def sample_group(question, cap, temperature, k, seed, mdl, tok, eos_ids):
    """k independent chains for one problem, in one generate call."""
    assert temperature > 0, (
        "greedy decoding from one prompt has one answer, so k copies of it would be one "
        "chain pretending to be k. This page's pools are all sampled.")
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids
    inp = torch.tensor([prompt_ids], device=mdl.device)
    torch.manual_seed(seed)
    out = mdl.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=cap,
        # State every decoding parameter: a checkpoint can ship sampling defaults in its
        # generation_config, and then the manifest would record a setup never configured.
        do_sample=True, temperature=temperature, top_p=GEN["top_p"], top_k=None,
        num_return_sequences=k,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        # The first end-of-turn token is where this chain ended; everything after it is
        # padding to the longest row in the batch and is not content it wrote.
        stop = next((i for i, t in enumerate(gen) if t in eos_ids), None)
        truncated = stop is None
        gen = gen[:stop] if stop is not None else gen[:cap]
        text = tok.decode(gen, skip_special_tokens=True)
        chains.append(dict(trace=text, own_tokens=len(gen), truncated=truncated))
    return chains


if RUN_MODE == "GENERATE":
    dev = get_device()
    got = harness.load_model(GEN["model_id"], device=dev, dtype=harness.pick_dtype(dev),
                             warmup_tokens=8)
    gmodel, gtok = got["model"], got["tokenizer"]
    harness.selftest(gtok)
    EOS_IDS = harness.end_of_turn_ids(gtok)

    # The item set is taken from page 7's records rather than re-derived from the dataset.
    # Pages 5 to 7 draw their 100 items by shuffling the band under a seed and slicing;
    # reimplementing that here would be one seed away from silently generating on a
    # DIFFERENT hundred problems, which would break every cross-page comparison the deck
    # rests on while every number still looked plausible. Reading the ids off the pool
    # makes the two item sets identical by construction rather than by agreement.
    src = {}
    for r in raw:
        src.setdefault(r["problem_id"], r)
    pool_items = [dict(unique_id=pid, problem=src[pid]["question"],
                       answer=src[pid]["ground_truth"], level=src[pid]["level"],
                       subject=src[pid]["subject"])
                  for pid in sorted(src)]
    assert len(pool_items) == GEN["n_items"], (
        f"the pool this notebook read carries {len(pool_items)} items and GEN asks for "
        f"{GEN['n_items']}. Set GEN['n_items'] to match rather than slicing, or the run "
        f"is on a different population from the page it feeds.")
    ITEM_SET_HASH = harness.sha256_of([p["unique_id"] for p in pool_items])

    # Identity with page 7 is by construction, and the check below says so rather than
    # comparing digests. Page 7 hashes its ids in the SHUFFLED order it drew them; this
    # hashes them sorted, so the two digests differ over the same hundred problems. A
    # digest comparison would fail loudly on a run that is perfectly correct, which is
    # worse than no check at all: the set comparison is the thing actually worth knowing.
    P7_MANIFEST = PAGE_07_DIR / "manifest.json"
    p7_ids = {r["problem_id"] for r in raw}
    assert {p["unique_id"] for p in pool_items} <= p7_ids, (
        "page 8's items are not a subset of page 7's, which cannot happen when they were "
        "read from page 7's own records. Something has rewritten the pool.")
    print(f"item set   {len(pool_items)} problems, sha256 {ITEM_SET_HASH[:12]} (sorted "
          f"order)")
    if P7_MANIFEST.exists():
        p7m = json.loads(P7_MANIFEST.read_text()).get("items", {})
        print(f"           taken from page 7's records, whose manifest records "
              f"{str(p7m.get('item_set_sha256'))[:12]} in draw order over n={p7m.get('n')}")
    OWN_RUN_ID = (f"p08-{GEN['dataset']}-n{len(pool_items)}-{ITEM_SET_HASH[:8]}"
                  f"-cap{GEN['chain_cap']}-p{GEN['top_p']}-{dev.type}")
    gstore = harness.TraceStore(OWN_TRACES, OWN_RUN_ID,
                               ("model_id", "config_id", "problem_id", "sample_index"))
    gstore.report()

    made = 0
    for temperature, pool_n in GEN["grid"]:
        cid = config_id(temperature, GEN["chain_cap"])
        bar = harness.progress(pool_items, desc=f"T={temperature} x{pool_n}", unit="item")
        for row in bar:
            pid = row["unique_id"]
            have = [gstore.get(GEN["model_id"], cid, pid, s) for s in range(pool_n)]
            if all(h is not None for h in have):
                continue
            for start in range(0, pool_n, GEN["group"]):
                size = min(GEN["group"], pool_n - start)
                if all(gstore.get(GEN["model_id"], cid, pid, start + j) is not None
                       for j in range(size)):
                    continue
                g_index = start // GEN["group"]
                seed = group_seed(pid, cid, g_index)
                for j, g in enumerate(sample_group(row["problem"], GEN["chain_cap"],
                                                   temperature, size, seed, gmodel, gtok,
                                                   EOS_IDS)):
                    extracted = extract_final_candidate(g["trace"])
                    gstore.append({
                        "model_id": GEN["model_id"], "config_id": cid, "problem_id": pid,
                        "sample_index": start + j, "group_index": g_index,
                        "batch_seed": seed, "batch_size": size,
                        "level": int(row["level"]), "subject": row["subject"],
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": temperature, "top_p": GEN["top_p"],
                        "chain_budget": GEN["chain_cap"], "trace": g["trace"],
                        "own_tokens": g["own_tokens"], "tokens_generated": g["own_tokens"],
                        "truncated": g["truncated"],
                        "extracted_answer": extracted,
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(grade_answer(extracted, row["answer"])),
                    })
                    made += 1
            bar.set_postfix(gen=made)
    del gmodel, got
    harness.free_memory()
    print(f"\n{made:,} chains generated into {OWN_TRACES}")
    print("Now set POOL_SOURCE = 'own' and RUN_MODE = 'SCORE', and run from the top.")
else:
    print(f"RUN_MODE = {RUN_MODE!r}: nothing is generated. This page's own pool is made "
          f"only in GENERATE, and only after the section 12 gate.")


## 16. Artifacts

`derived.json` is the contract with the page: the aggregates it plots, the claims it may make, and
the manifest that says what produced them. The candidates are a download rather than a bundle.


In [ ]:
manifest = {
    "score_id": SCORE_ID,
    "notebook": "page_08_best_of_n.ipynb",
    "notebook_sha256": harness.file_sha256("page_08_best_of_n.ipynb"),
    "harness": harness.fingerprint(),
    "page": 8, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "candidates": {"source": POOL_ORIGIN, "run_id": SOURCE_RUN_ID, "model_id": MODEL_ID,
                   "configs": [CONFIG_META[c] for c in LADDER_CONFIGS],
                   "chain_cap": CAP,
                   "n_items_at": {c: len(ITEMS_AT[c]) for c in LADDER_CONFIGS},
                   "grid_items": len(GRID_ITEMS), "main_config": MAIN_CONFIG,
                   "n_candidates": sum(len(POOLS[c][p]) for c in LADDER_CONFIGS
                                       for p in ITEMS_AT[c])},
    "verifiers": [dict(name=v["name"], kind=v["kind"],
                       model=MODEL_ID if v["name"] == "self" else v["model"],
                       quant=v.get("quant"),
                       loaded_as=("rebuilt on transformers' Qwen2 body plus the "
                                  "checkpoint's value head; the checkpoint's own modelling "
                                  "code returns NaN under this transformers version"
                                  if v["kind"] == "rm" else "as published"),
                       scored_at=("the final token, one value for the whole solution "
                                  "(outcome, not process)" if v["kind"] == "rm" else None),
                       solves_these_items=v.get("solves"),
                       solves_source=("measured in this run"
                                      if v["name"] in JUDGE_SOLVES_MEASURED
                                      else "unmeasured" if v.get("solves") is None
                                      else "page 6"),
                       stronger_generator_than_the_writer=(
                           v.get("solves") is not None and WRITER_SOLVES is not None
                           and v["solves"] > WRITER_SOLVES),
                       complete=any(h["name"] == v["name"] for h in HAVE),
                       scored_this_run=SCORED_THIS_RUN.get(v["name"], 0),
                       prompt=JUDGE_PROMPT if v["kind"] == "yesno" else None)
                  for v in ACTIVE],
    "subsampling": {"n_resamples": N_RESAMPLES, "seed": SEED,
                    "rule": "subsets without replacement, identical across selectors",
                    "score_tie_break": "lowest sample index",
                    "intervals": f"cluster bootstrap over items, {N_BOOTSTRAP} resamples"},
    "buckets": {"config_id": BUCKET_CONFIG, "holdout": BUCKET_HOLDOUT,
                "measuring_chains": BUCKET_N, "base_rate_if_circular": FULL_PILE_RATE},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    # Read before any number on this page.
    "claims": {
        # The page's whole thesis. False means it may not be claimed.
        # THE page's claim, and the only one of these that blocks: scoring N candidates
        # and keeping the best beats taking one, resolved by a paired test on the same
        # problems. This is ch05's comparison; the vote is ch04 and is not part of it.
        "best_of_n_beats_one_chain_resolved": bool(
            BEST_VS_ONE and BEST_VS_ONE["resolved"]),
        # Reported, never blocking. A strictly harder question, and one the book never
        # asks: it needs page 7's candidates to be askable at all.
        "best_of_n_beats_counting": BEATS_COUNTING,
        "best_of_n_beats_counting_resolved": bool(
            BEST_VS_MAJ and BEST_VS_MAJ["resolved"]),
        "best_of_n_beats_counting_without_a_stronger_generator": BEATS_COUNTING_CLEAN,
        "best_of_n_beats_one_chain": BEATS_COT,
        "verifier_scored": bool(HAVE),
        "bar_falls_with_temperature": bool(
            len(LADDER_CONFIGS) > 1
            and BAR[LADDER_CONFIGS[-1]][-1]["bar"] < BAR[LADDER_CONFIGS[0]][-1]["bar"]),
        # WHY the bar falls, which is the difference between an opportunity and a smaller
        # room. False here means diversity bought no extra reach, only a weaker vote.
        "coverage_rises_with_temperature": bool(
            len(LADDER_CONFIGS) > 1
            and at_n(LADDER_CONFIGS[-1], "coverage") > at_n(LADDER_CONFIGS[0], "coverage")),
        "trap_bucket_plottable": TRAP_OK,
        "majority_declines_on_traps": bool(MAJORITY_DECLINES_ON_TRAPS),
        "buckets_are_held_out": BUCKET_HOLDOUT,
        "heuristic_is_a_length_ranking": HEURISTIC_IS_LENGTH,
        # False means the logprob diagnostic is mostly recovering the greedy chain.
        "logprob_is_more_than_greedy": (GREEDY_AGREEMENT is None
                                        or GREEDY_AGREEMENT["answer_agreement"] <= 0.8),
        "agrees_with_page_7": (PAGE_7_AGREEMENT is None
                               or not PAGE_7_AGREEMENT.get("same_candidates")
                               or abs(PAGE_7_AGREEMENT["vote_delta_pts"]) <= 0.5),
        "worked_case_exists": bool(WORKED),
        # Whether the judge ladder can be read as a ranking at all.
        "judge_ranking_resolved": bool(RANKING_RESOLVED),
        # The ablation's own claim: a judge WORSE at the problems than the writer still
        # beats counting, which is "verification is easier than generation" measured.
        "a_weaker_judge_beats_counting": bool(
            CHEAPEST_SUFFICIENT and WRITER_SOLVES
            and CHEAPEST_SUFFICIENT["solves"] < WRITER_SOLVES),
    },
    "gate": GATE,
    "paired": PAIRED,
    "bar": {c: BAR[c] for c in LADDER_CONFIGS},
    "bar_paired": {c: BAR_GRID[c] for c in LADDER_CONFIGS},
    "figure_1": {"config_id": FIG1_CONFIG, "axis": LADDERS[FIG1_CONFIG]["axis"],
                 "headline_selector": HEADLINE_NAME,
                 "lines": [{"selector": n, "what": w,
                            "points": LADDERS[FIG1_CONFIG]["ladder"][n]}
                           for n, w, _ in HEADLINE],
                 "specimen": SPECIMEN_BUNDLE},
    "figure_2": {"grid_n": GRID_N, "grid_items": len(GRID_ITEMS),
                 "paired": True,
                 "by_temperature": [dict(CONFIG_META[c],
                                         **{k: at_n(c, k) for k in
                                            ("pass_at_1", "majority", "coverage", "bar",
                                             "bar_lo", "bar_hi")},
                                         best_of_n=(best_of_n_at(c, ladders=LADDERS_GRID)[1]
                                                    ["accuracy"] if BON_NAMES else None))
                                    for c in LADDER_CONFIGS],
                 "buckets": {"config_id": BUCKET_CONFIG, "base_rate": BASE_RATE,
                             "axis": BUCKET_AXIS,
                             "slices": {b: dict(n_items=d["n_items"], share=d["share"],
                                                ladder=d["ladder"])
                                        for b, d in BUCKET_LADDER.items()}}},
    "figure_3": {"config_id": FIG3_CONFIG, "bar": bar_here, "reliability": RELIABILITY,
                 "judge_ladder": JUDGE_LADDER,
                 "reliability_across_temperatures": RELIABILITY_ACROSS,
                 "judge_solves_measured": JUDGE_SOLVES_MEASURED,
                 "writer_solves": WRITER_SOLVES,
                 "cheapest_sufficient_judge": CHEAPEST_SUFFICIENT,
                 "discrimination": DISCRIMINATION, "worked_case": WORKED},
    "ladders": {c: LADDERS[c] for c in LADDER_CONFIGS},
    "ladders_paired": {c: LADDERS_GRID[c] for c in LADDER_CONFIGS},
    "page_7_agreement": PAGE_7_AGREEMENT,
    "logprob_vs_greedy": GREEDY_AGREEMENT,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"derived.json   {harness.kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
if SCORES_PATH.exists():
    print(f"scores.jsonl   {harness.kb(SCORES_PATH):>10,.1f} KB   <- published as a download")

# The blocking claim is the CLEAN one: a win that needed a judge better at the problems
# than the model being judged does not settle what this page says it settles.
# What must hold for the page to exist: a verifier was run, its gain over one chain is
# resolved by a paired test, and the vote here reproduces page 7's. Beating COUNTING is
# deliberately absent -- see section 12 for why a harder, self-imposed question does not
# get to veto a result the page does claim.
blocking = ["verifier_scored", "best_of_n_beats_one_chain_resolved", "agrees_with_page_7"]
failed = [k for k in blocking if not derived["claims"][k]]
print(f"\n{'TRANSLATABLE' if not failed else 'NOT TRANSLATABLE: ' + ', '.join(failed)}")
for k, v in derived["claims"].items():
    print(f"   {'ok ' if v else 'NO '} {k}")
print("\nThe blocking three are the page's thesis and its consistency with page 7. The rest")
print("change what the page SAYS rather than whether it can be written.")


## 17. Notes for re-derivation

- **The bar is measured before any verifier is loaded.** `majority(N) / coverage(N)` is fixed in
  section 3, and every later claim is checked against it. A threshold chosen after seeing the
  result is not a threshold.
- **The N axis is subsampled from a fixed-depth pool.** Page 7's N axis changes chain length as it
  changes N, because its budget is matched; this one does not, or the ladder would be measuring
  page 7's variable.
- **Every rule sees the same subsets**, drawn once per (item, configuration, N, repeat). Comparing
  rules across different draws would put sampling noise into the spread the figure reads as a
  difference between rules.
- **Intervals are over items, not over draws.** The draws inside one item are subsets of one pile
  and are not independent; treating them as independent trials makes the interval several times too
  narrow.
- **Reliability is defined only on items where a correct candidate exists.** On hopeless items every
  rule is wrong by construction, and including them would compress the differences the figure exists
  to show.
- **`self` is the control, not a contender.** A model grading its own work shares every error mode
  that produced it, and if it scores near the bar that is worth a sentence rather than a curve.
- **A judge that is a stronger generator proves the wrong thing.** Every tier records its own
  measured accuracy on these items, and the gate reports the best rung that is *not* stronger than
  the writer separately from the best rung overall. The first is the page's claim; the second is a
  ceiling.
- **A same-family judge is a weaker test than it looks.** All three tiers share Qwen2.5
  pretraining, so a judge that lands close to the self-judge is ambiguous between "size does not
  help" and "this training does not help", and settling that needs another family. The trained
  reward model tier is also the answer to this, since it is trained for a different job rather than
  scaled up at the same one.
- **The heuristic's weights are ch05's defaults** and are in the manifest. Changing them changes
  what "grades on presentation" means, so change them in the knobs and say so, or not at all.
- Ties among scores break to the lowest sample index. The vote's ties are not broken, because page 7
  does not break them.
- Temperature sampling is not reproducible across machines, so **the transcripts are the
  provenance**: the analysis is deterministic from them, and section 13 recomputes page 7's headline
  from the same file page 7 wrote.

What the page may claim is [`docs/page_8_measured_payoff.md`](../docs/page_8_measured_payoff.md),
checked at translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) section 6).
